# SynthTextEval evaluation pipeline for ICL-PILOT

> Previous-study snapshot frozen on TranscendBU. This interface retains reusable code; the revised study protocol is pending. On 20 October 2026, story counterbalancing was revised and the generator was corrected to a pinned official Llama 3.1 8B Instruct checkpoint. Other protocol choices require review. Evaluation inputs require explicit configuration. Historical execution outputs have been archived and cleared.

Results are written to a fresh `results/framework_based_evaluation/run_<UTC timestamp>/` directory. Existing run names are refused. The previous study and its saved notebook outputs are frozen externally; see the archive index under `docs/provenance/`. The two revised-study input paths in `configs/evaluation/inputs.json` are pending. Evaluation stops before creating a result directory until both are supplied.

This notebook adapts the [SynthTextEval](https://github.com/kr-ramesh/synthtexteval) pipeline to the traceable ENNI-style narratives in ICL-PILOT. It evaluates generated versus real text with corpus-level descriptive and automated text-quality measures, downstream TD/SLI classification, age-cohort performance/fairness auditing, and reference-overlap privacy diagnostics.

The downstream-classification analysis uses rotating pair-level splits. **Predictions are evaluated within each training round and only then summarized across rounds**; repeated predictions of the same underlying child across different trained models are not treated as independent test observations. Split membership uses the composite `(cohort_year, pair_id)` key throughout, so reused `pair_id` values across cohorts cannot contaminate a split.

The balanced descriptive/text-quality comparison controls cohort and TD/SLI composition, but the selected synthetic and real rows are distributional samples rather than semantic pairs. The privacy section instead uses the full available synthetic corpus and full available real reference corpus, because a balanced subsample would undercount possible entity/context overlap.

The package's entity-leakage and entity-phrase-overlap analyses remain **reference-overlap diagnostics**, not verified privacy-leakage estimates: no supplied list of protected identifiers exists for these de-identified stories. Canary attacks are not run because the generator was not trained with planted canaries. The coreference module remains out of scope because the export has no gold coreference annotation.

At the end, the notebook writes a reproducibility manifest, copies a compact result bundle to Google Drive, creates and verifies a ZIP, and starts a local download. Runtime unassignment is deliberately placed in the final cell and requires explicit confirmation that the browser download has completed.


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module=r"jupyter_client.*",
)


In [ ]:
# Repository paths and run configuration.
from pathlib import Path
import sys

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
COLAB_REPO_DIR = Path("/content/ICL-PILOT")
candidate_roots = [COLAB_REPO_DIR, Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((path for path in candidate_roots if (path / "pyproject.toml").exists()), None)

if PROJECT_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Run this notebook from the ICL-PILOT repository or open it in Colab.")
    !git clone {REPO_URL} {COLAB_REPO_DIR}
    PROJECT_ROOT = COLAB_REPO_DIR

%cd {PROJECT_ROOT}
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from icl_pilot.evaluation.runs import create_framework_evaluation_run
from icl_pilot.evaluation.inputs import load_evaluation_inputs

EVALUATION_INPUT_CONFIG = PROJECT_ROOT / "configs/evaluation/inputs.json"
EVALUATION_RUN_ID = None  # None creates a UTC timestamp; an explicit name must be unused.
TEXT_COLUMN = "story_text"
SOURCE_COLUMN = "source"
GROUP_COLUMN = "group"
COHORT_COLUMN = "cohort_year"
COMPARISON_COHORTS = (4, 5, 6, 7)
COMPARISON_GROUPS = ("TD", "SLI")
SAMPLE_SEED = 42
MAX_SAMPLES_PER_GROUP = None  # Set to 20 for a CPU smoke test.

RUN_MAUVE = True
RUN_REPEATED_MAUVE = True
REPEATED_MAUVE_N_RUNS = 20
MAUVE_STABILITY_BASE_SEED = 25  # MAUVE library default; first repeated run should reproduce the wrapper score.
MAUVE_DEVICE_ID = 0  # Fallback only; repeated diagnostics inherit the main SynthTextEval MAUVE device.
MAUVE_MAX_TEXT_LENGTH = 1024  # Fallback only; repeated diagnostics inherit the main SynthTextEval value.
# In addition to varying MAUVE's seed on the full balanced corpus, run an 80%
# subsampling sensitivity analysis. Set to None to disable the subsampling arm.
MAUVE_SUBSAMPLE_FRACTION = 0.80

RUN_CLASSIFICATION = True  # Requires a CUDA-enabled Colab runtime.
RUN_GILLAM_EXTERNAL_TEST = True  # Gillam remains test-only.
RUN_CHILD_LEVEL_CLASSIFICATION = True  # Aggregate ENNI story probabilities to one prediction per child.
RUN_PRIVACY_ANALYSIS = True
PRIVACY_ENTITY_LABELS = ("PERSON", "ORG", "GPE", "LOC", "FAC", "NORP")
PRIVACY_MAX_CONTEXT_WINDOW = 4
CLASSIFICATION_ROUNDS = ("round_01", "round_02", "round_03", "round_04")
CLASSIFIER_MODEL = "bert-base-uncased"
LABEL_MAPPING = {"SLI": 0, "TD": 1}
META_RESPONSE_MARKERS = ("I replaced the main characters to match the target story",)

# Export settings. Model checkpoints are large and are excluded from the portable
# ZIP by default; predictions, metrics, split definitions, manifests, and plots are kept.
DRIVE_EXPORT_ROOT = Path("/content/drive/MyDrive/ICL-PILOT/framework_based_evaluation")
EXPORT_INCLUDE_MODEL_CHECKPOINTS = False
EXPORT_ARCHIVE_PREFIX = "framework_based_evaluation"

evaluation_inputs = load_evaluation_inputs(PROJECT_ROOT, config_path=EVALUATION_INPUT_CONFIG)
TEXT_RECORDS_CSV = evaluation_inputs["text_records_csv"]
GILLAM_EXTERNAL_TEXT_CSV = evaluation_inputs["gillam_external_text_csv"]
RESULTS_DIR = create_framework_evaluation_run(PROJECT_ROOT, run_id=EVALUATION_RUN_ID)
print({
    "project_root": str(PROJECT_ROOT),
    "text_records": str(TEXT_RECORDS_CSV),
    "gillam_external_records": str(GILLAM_EXTERNAL_TEXT_CSV),
    "results_dir": str(RESULTS_DIR),
    "evaluation_run_id": RESULTS_DIR.name,
})


In [ ]:
# Install SynthTextEval from a local clone and apply narrowly scoped compatibility fixes.
# A sentinel prevents a reinstall loop after the one required runtime restart.
from pathlib import Path
import re
import subprocess
import sys

SYNTHTEXTEVAL_REVISION = "3d63a172418a985fd0d997e5c993f6a0390c294e"
SYNTHTEXTEVAL_DIR = Path("/content/synthtexteval-src")
INSTALL_SENTINEL = SYNTHTEXTEVAL_DIR / ".icl_pilot_install_ready"


def run_checked(command):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(part) for part in command], check=True)


if not (SYNTHTEXTEVAL_DIR / ".git").exists():
    run_checked(["git", "clone", "https://github.com/kr-ramesh/synthtexteval.git", SYNTHTEXTEVAL_DIR])

sentinel_matches = (
    INSTALL_SENTINEL.exists()
    and INSTALL_SENTINEL.read_text(encoding="utf-8").strip() == SYNTHTEXTEVAL_REVISION
)

if not sentinel_matches:
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "fetch", "--all", "--tags", "-q"])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "checkout", "-q", SYNTHTEXTEVAL_REVISION])
    run_checked(["git", "-C", SYNTHTEXTEVAL_DIR, "reset", "--hard", "-q", SYNTHTEXTEVAL_REVISION])

    # Packaging fix: this repository revision lacks synthtexteval/eval/__init__.py,
    # while setup.py relies on package discovery.
    eval_init = SYNTHTEXTEVAL_DIR / "synthtexteval" / "eval" / "__init__.py"
    eval_init.touch(exist_ok=True)

    # Transformers compatibility fix. Keep tokenization semantics unchanged:
    # encode_plus -> tokenizer(...) and pad_to_max_length=True -> padding="max_length".
    classifier_utils_path = (
        SYNTHTEXTEVAL_DIR
        / "synthtexteval"
        / "eval"
        / "downstream"
        / "classify"
        / "utils.py"
    )
    utils_text = classifier_utils_path.read_text(encoding="utf-8")
    utils_text, encode_plus_count = re.subn(
        r"tokenizer\s*\.\s*encode_plus\s*\(",
        "tokenizer(",
        utils_text,
    )
    utils_text, legacy_padding_count = re.subn(
        r"pad_to_max_length\s*=\s*True",
        'padding="max_length"',
        utils_text,
    )
    classifier_utils_path.write_text(utils_text, encoding="utf-8")

    assert re.search(r"tokenizer\s*\.\s*encode_plus\s*\(", utils_text) is None
    assert "pad_to_max_length" not in utils_text
    assert 'padding="max_length"' in utils_text

    print({
        "synthtexteval_revision": SYNTHTEXTEVAL_REVISION,
        "encode_plus_replacements": encode_plus_count,
        "legacy_padding_replacements": legacy_padding_count,
        "source": str(SYNTHTEXTEVAL_DIR),
    })

    # Install repository requirements first, then pin Transformers last so an unpinned
    # transformers requirement cannot leave a v5 installation active.
    run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "SynthTextEval"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "-r", SYNTHTEXTEVAL_DIR / "requirements.txt"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "pyLDAvis", "accelerate>=0.26.0"])
    run_checked([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SYNTHTEXTEVAL_DIR])
    run_checked([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

    INSTALL_SENTINEL.write_text(SYNTHTEXTEVAL_REVISION + "\n", encoding="utf-8")
    SYNTH_INSTALL_RESTART_REQUIRED = True
    print("\nSynthTextEval installation/patch completed.")
    print("Restart the runtime ONCE now, then rerun the notebook from the top.")
else:
    SYNTH_INSTALL_RESTART_REQUIRED = False
    print("Pinned SynthTextEval installation already prepared in this runtime; reinstall skipped.")


In [ ]:
# Run after restarting the kernel/runtime. Verify that Python is using the patched clone.
from pathlib import Path
import inspect
import synthtexteval
import transformers

from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs
from synthtexteval.eval.downstream.classify import utils as classifier_utils
from synthtexteval.eval.text_quality import metrics as text_quality_metrics

loaded_utils_path = Path(classifier_utils.__file__).resolve()
expected_source_root = Path("/content/synthtexteval-src").resolve()

print("SynthTextEval:", synthtexteval.__file__)
print("Transformers:", transformers.__version__)
print("Classifier utils:", loaded_utils_path)

assert expected_source_root in loaded_utils_path.parents, (
    f"Wrong SynthTextEval copy loaded: {loaded_utils_path}. Expected the editable clone under {expected_source_root}."
)
assert transformers.__version__.startswith("4."), "This notebook expects Transformers 4.x."

tokenize_source = inspect.getsource(classifier_utils.tokenize_data)
assert "encode_plus" not in tokenize_source, "Old encode_plus call is still active."
assert "pad_to_max_length" not in tokenize_source, "Old pad_to_max_length argument is still active."
assert "padding" in tokenize_source and "max_length" in tokenize_source

# The public README uses QualEval, but tolerate an older class name if a repository
# revision exposes it instead. Cell 17 uses the same resolution logic.
quality_class = getattr(text_quality_metrics, "QualEval", None)
if quality_class is None:
    quality_class = getattr(text_quality_metrics, "TextQualityEval", None)
assert quality_class is not None, (
    "No supported text-quality evaluator class (QualEval/TextQualityEval) was found in "
    f"{text_quality_metrics.__file__}."
)
print("Text-quality evaluator:", quality_class.__name__)
print("SynthTextEval installation/import checks passed.")


In [ ]:
import hashlib
import json
import pandas as pd


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


text_records = pd.read_csv(TEXT_RECORDS_CSV)
required_columns = {
    SOURCE_COLUMN, GROUP_COLUMN, COHORT_COLUMN, TEXT_COLUMN, "pair_id", "round_id",
    "source_child_id", "td_child_id", "sli_child_id",
}
missing_columns = required_columns.difference(text_records.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"
assert not text_records[TEXT_COLUMN].isna().any()
assert (text_records[TEXT_COLUMN].astype(str).str.strip() != "").all()

# Preserve the raw export and make the analytic exclusion reproducible. This excludes only
# the known LLM self-commentary, never human ENNI text.
meta_response_mask = pd.Series(False, index=text_records.index)
for marker in META_RESPONSE_MARKERS:
    meta_response_mask |= text_records[TEXT_COLUMN].str.contains(marker, case=False, regex=False, na=False)
meta_response_mask &= text_records[SOURCE_COLUMN].eq("synthetic")
excluded_records = text_records.loc[meta_response_mask].copy()
analysis_records = text_records.loc[~meta_response_mask].copy()
excluded_records.to_csv(RESULTS_DIR / "excluded_meta_generations.csv", index=False)

summary = {
    "raw_records": int(len(text_records)),
    "analytic_records": int(len(analysis_records)),
    "excluded_meta_generations": int(len(excluded_records)),
    "exclusion_markers": list(META_RESPONSE_MARKERS),
    "text_records_sha256": sha256_file(TEXT_RECORDS_CSV),
    "gillam_external_text_sha256": sha256_file(GILLAM_EXTERNAL_TEXT_CSV),
}
(RESULTS_DIR / "input_qc_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(summary)

show_columns = [
    col for col in ["source_csv", "source_row_number", "pair_id", "story_id", "stage"]
    if col in excluded_records.columns
]
if show_columns:
    display(excluded_records[show_columns])


In [ ]:
# Balanced, independently sampled corpus inputs for descriptive and text-quality modules.
# Row i in the synthetic table is NOT assumed to be semantically paired with row i in the real table.
manifest_columns = [
    col for col in [
        TEXT_COLUMN, COHORT_COLUMN, GROUP_COLUMN, "source_child_id", "pair_id", "round_id",
        "source_csv", "source_row_number", "story_id", "stage",
    ]
    if col in analysis_records.columns
]

synthetic_parts, reference_parts = [], []
for cohort_year in COMPARISON_COHORTS:
    for group in COMPARISON_GROUPS:
        synthetic_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "synthetic")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            manifest_columns,
        ]
        reference_group = analysis_records.loc[
            (analysis_records[SOURCE_COLUMN] == "real")
            & (analysis_records[COHORT_COLUMN] == cohort_year)
            & (analysis_records[GROUP_COLUMN] == group),
            manifest_columns,
        ]
        n = min(len(synthetic_group), len(reference_group))
        if MAX_SAMPLES_PER_GROUP is not None:
            n = min(n, MAX_SAMPLES_PER_GROUP)
        assert n > 0, f"No comparable {cohort_year}-year-old {group} text records found."
        seed = SAMPLE_SEED + 100 * cohort_year + (0 if group == "TD" else 1)
        synthetic_parts.append(synthetic_group.sample(n=n, random_state=seed).reset_index(drop=True))
        reference_parts.append(reference_group.sample(n=n, random_state=seed).reset_index(drop=True))

synthetic_texts = pd.concat(synthetic_parts, ignore_index=True)
reference_texts = pd.concat(reference_parts, ignore_index=True)
comparison_df = pd.DataFrame({
    "source": synthetic_texts[TEXT_COLUMN].astype(str).tolist(),
    "reference": reference_texts[TEXT_COLUMN].astype(str).tolist(),
})

# Save exact sampled inputs, not only group counts, so the distributional analyses are reproducible.
synthetic_texts.to_csv(RESULTS_DIR / "balanced_synthetic_sample.csv", index=False)
reference_texts.to_csv(RESULTS_DIR / "balanced_real_sample.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "balanced_text_quality_input.csv", index=False)

comparison_manifest = pd.concat(
    [
        synthetic_texts.drop(columns=[TEXT_COLUMN], errors="ignore").add_prefix("synthetic_"),
        reference_texts.drop(columns=[TEXT_COLUMN], errors="ignore").add_prefix("reference_"),
    ],
    axis=1,
)
comparison_manifest.to_csv(RESULTS_DIR / "balanced_corpus_sample_manifest.csv", index=False)
print({"synthetic_texts": len(synthetic_texts), "reference_texts": len(reference_texts)})
display(
    synthetic_texts.groupby([COHORT_COLUMN, GROUP_COLUMN]).size().rename("synthetic_n").reset_index()
    .merge(
        reference_texts.groupby([COHORT_COLUMN, GROUP_COLUMN]).size().rename("real_n").reset_index(),
        on=[COHORT_COLUMN, GROUP_COLUMN],
        how="outer",
    )
)


## Descriptive analysis

These are generic corpus-level comparisons. The matching is balanced by cohort and literal corpus group, but row order does not define a child-, prompt-, or story-level pair.

In [ ]:
import os
import importlib.util
import synthtexteval

print("cwd:", os.getcwd())
print("module:", synthtexteval)
print("file:", getattr(synthtexteval, "__file__", None))
print("path:", list(getattr(synthtexteval, "__path__", [])))
print("eval spec:", importlib.util.find_spec("synthtexteval.eval"))

assert importlib.util.find_spec("synthtexteval.eval") is not None, (
    "synthtexteval.eval is still unavailable. Restart the runtime and rerun "
    "the installation cells from the top."
)


In [ ]:
import io
from contextlib import redirect_stdout
import nltk
import pandas as pd
from synthtexteval.eval.descriptive.descriptor import TextDescriptor
from synthtexteval.eval.descriptive.arguments import TextDescriptorArgs

nltk.download("punkt_tab")

synth_df = comparison_df.rename(columns={"source": "output_text"})[["output_text"]].copy()
real_texts = comparison_df["reference"].tolist()
descriptor_args = TextDescriptorArgs(
    pkl_file_path=str(RESULTS_DIR / "entity-output.pkl"),
    plt_file_path=str(RESULTS_DIR / "entity-analysis.png"),
    produce_plot=True,
)
desc_analyze = TextDescriptor(
    texts=synth_df["output_text"].tolist(),
    args=descriptor_args,
    reference_texts=real_texts,
)

# The package prints these metrics rather than consistently returning a structured object.
# Capture the exact printed output while still showing it in the notebook and save it.
descriptive_stdout = io.StringIO()
with redirect_stdout(descriptive_stdout):
    descriptive_result = desc_analyze._compare_to_reference_distribution(
        metrics=["text-length", "jaccard", "cosine"]
    )
descriptive_text = descriptive_stdout.getvalue()
print(descriptive_text, end="")
(RESULTS_DIR / "descriptive_distribution_comparison.txt").write_text(
    descriptive_text, encoding="utf-8"
)
if descriptive_result is not None:
    (RESULTS_DIR / "descriptive_distribution_return_value.txt").write_text(
        repr(descriptive_result) + "\n", encoding="utf-8"
    )

# Optional interactive topic display. This describes the synthetic corpus; it should not
# be interpreted as a real-vs-synthetic topic-distribution comparison by itself.
import pyLDAvis
pyLDAvis.enable_notebook()
topic_model = desc_analyze._topic_modeling_display(num_topics=3)
if topic_model is not None:
    try:
        pyLDAvis.save_html(topic_model, str(RESULTS_DIR / "synthetic_lda_topics.html"))
    except Exception as exc:
        print("Could not save LDA HTML:", exc)
topic_model


## Downstream utility: rotating synthetic-to-real TD/SLI classification

For each usable generation round, the notebook derives pair membership from the IDs embedded in the generated artifacts. Synthetic descendants of prompt pairs and real prompt-pair narratives form the training material; one whole prompt pair per cohort is reserved for validation; real narratives from all other paired children form the test set. Split membership uses `(cohort_year, pair_id)` rather than bare `pair_id`, and train/validation/test child sets are asserted to be disjoint within each round.

The stock SynthTextEval random split helper is deliberately not used because it could split related prompt-pair descendants across partitions. The notebook still uses SynthTextEval's classifier and fairness utilities.

The four trained round-models are treated as repeated rotating-split evaluations. A child may legitimately be tested by more than one round-model, so **raw predictions from different rounds are never pooled to create an inflated sample size**. Accuracy/F1 and cohort fairness/performance are computed within each training round and then summarized across rounds. The reported SD therefore describes variation across the rotating trained models; it is not an independence-based standard error or confidence interval.

The same trained models can additionally be tested on Gillam as a strictly external corpus. Gillam contributes no training or validation rows, and an ID-disjointness assertion is enforced. Each Gillam story is evaluated once per trained round-model; external metrics are therefore computed per model and summarized across models rather than pooling repeated model predictions. Both Gillam and ENNI use one narrative per row, while Gillam remains an external-transfer test across corpora.


In [ ]:
# Build round-specific datasets from generated-artifact IDs using the composite
# (cohort_year, pair_id) key throughout.
CLASSIFICATION_DIR = RESULTS_DIR / "classification"
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)

synthetic_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("synthetic")].copy()
real_source = analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("real")].copy()
for column in ("source_child_id", "td_child_id", "sli_child_id"):
    synthetic_source[column] = pd.to_numeric(synthetic_source[column], errors="raise").astype("int64")
real_source["source_child_id"] = pd.to_numeric(real_source["source_child_id"], errors="raise").astype("int64")


def add_pair_key(frame):
    frame = frame.copy()
    frame["_pair_key"] = list(zip(
        frame[COHORT_COLUMN].astype(int),
        frame["pair_id"].astype(str),
    ))
    return frame


synthetic_source = add_pair_key(synthetic_source)

# Gillam is a separate external test source. Each row is one @G-bounded story;
# it is never concatenated with analysis_records or any training frame.
gillam_records = pd.read_csv(GILLAM_EXTERNAL_TEXT_CSV)
gillam_required = {TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "story_id", "source", "sex"}
gillam_missing = gillam_required.difference(gillam_records.columns)
assert not gillam_missing, f"Missing Gillam columns: {sorted(gillam_missing)}"
assert gillam_records["source"].eq("real_external").all()
assert set(gillam_records[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
assert not gillam_records.duplicated(["source_child_id", "story_id"]).any()
assert gillam_records[TEXT_COLUMN].notna().all()
gillam_ids = set(gillam_records["source_child_id"].astype(str))
enni_ids = set(real_source["source_child_id"].astype(str))
external_id_overlap = sorted(gillam_ids.intersection(enni_ids))
assert not external_id_overlap, f"Gillam external IDs occur in ENNI data: {external_id_overlap}"

external_columns = [TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "story_id", "source_dataset", "sex"]
external_gillam_test = gillam_records[external_columns].copy()
external_gillam_test = external_gillam_test.rename(columns={TEXT_COLUMN: "output_text"})
external_gillam_test["Label"] = external_gillam_test[GROUP_COLUMN].map(LABEL_MAPPING)
external_gillam_test["classification_round"] = "external_gillam"  # dataset identifier, not model round
external_gillam_test["pair_id"] = pd.NA
assert external_gillam_test["Label"].notna().all()

EXTERNAL_GILLAM_DIR = CLASSIFICATION_DIR / "external_gillam"
EXTERNAL_GILLAM_DIR.mkdir(parents=True, exist_ok=True)
external_gillam_test.to_csv(EXTERNAL_GILLAM_DIR / "test.csv", index=False)
external_gillam_audit = {
    "status": "prepared_test_only",
    "rows": int(len(external_gillam_test)),
    "unique_children": int(external_gillam_test["source_child_id"].nunique()),
    "group_counts": external_gillam_test[GROUP_COLUMN].value_counts().to_dict(),
    "enni_id_overlap": len(external_id_overlap),
    "document_unit": "one Gillam @G-bounded story",
    "training_use": False,
}
(EXTERNAL_GILLAM_DIR / "input_audit.json").write_text(json.dumps(external_gillam_audit, indent=2) + "\n")
print({"external_gillam": external_gillam_audit})

pair_metadata = synthetic_source[[COHORT_COLUMN, "pair_id", "td_child_id", "sli_child_id", "_pair_key"]].drop_duplicates()
assert pair_metadata.groupby([COHORT_COLUMN, "pair_id"]).size().eq(1).all(), "Ambiguous generated pair metadata."
pair_rounds = synthetic_source[[COHORT_COLUMN, "pair_id", "round_id", "_pair_key"]].drop_duplicates()
assert pair_rounds.groupby([COHORT_COLUMN, "pair_id"])["round_id"].nunique().eq(1).all(), "A generated pair occurs in multiple rounds."

# Audit whether raw pair_id values are reused across cohorts. Reuse is safe because all
# split operations below use _pair_key=(cohort_year, pair_id).
pair_id_cohort_counts = pair_metadata.groupby("pair_id")[COHORT_COLUMN].nunique().rename("n_cohorts").reset_index()
pair_id_cohort_counts.to_csv(CLASSIFICATION_DIR / "pair_id_cross_cohort_audit.csv", index=False)
print("pair_id values reused across cohorts:", int((pair_id_cohort_counts["n_cohorts"] > 1).sum()))

sli_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "_pair_key", "sli_child_id"]].rename(columns={"sli_child_id": "source_child_id"})
sli_pair_children[GROUP_COLUMN] = "SLI"
td_pair_children = pair_metadata[[COHORT_COLUMN, "pair_id", "_pair_key", "td_child_id"]].rename(columns={"td_child_id": "source_child_id"})
td_pair_children[GROUP_COLUMN] = "TD"
pair_children = pd.concat([sli_pair_children, td_pair_children], ignore_index=True)
assert not pair_children.duplicated([COHORT_COLUMN, "pair_id", GROUP_COLUMN]).any()

real_columns = [TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id"]
real_source = real_source[real_columns].copy()


def real_rows_for_pair_keys(pair_keys):
    pair_lookup = pair_children.loc[
        pair_children["_pair_key"].isin(pair_keys),
        [COHORT_COLUMN, GROUP_COLUMN, "source_child_id", "pair_id", "_pair_key"],
    ].copy()
    rows = real_source.merge(
        pair_lookup,
        on=[COHORT_COLUMN, GROUP_COLUMN, "source_child_id"],
        how="inner",
        validate="many_to_one",
    )
    return rows


def child_keys(frame):
    return set(zip(
        frame[COHORT_COLUMN].astype(int),
        frame[GROUP_COLUMN].astype(str),
        frame["source_child_id"].astype(int),
    ))


def classifier_frame(frame, classification_round):
    output = frame[[TEXT_COLUMN, GROUP_COLUMN, COHORT_COLUMN, "source_child_id", "pair_id"]].copy()
    output = output.rename(columns={TEXT_COLUMN: "output_text"})
    output["Label"] = output[GROUP_COLUMN].map(LABEL_MAPPING)
    output["classification_round"] = classification_round
    assert output["Label"].notna().all()
    return output


prepared_rounds, split_audit = {}, []
for round_id in CLASSIFICATION_ROUNDS:
    round_synthetic = synthetic_source.loc[synthetic_source["round_id"].eq(round_id)].copy()
    if round_synthetic.empty:
        split_audit.append({"round_id": round_id, "status": "skipped_no_generated_rows"})
        continue

    train_pair_keys, validation_pair_keys, test_pair_keys = set(), set(), set()
    usable_cohorts = []

    for cohort_year in sorted(round_synthetic[COHORT_COLUMN].unique()):
        prompt_pair_keys = sorted(set(
            round_synthetic.loc[round_synthetic[COHORT_COLUMN].eq(cohort_year), "_pair_key"]
        ))
        all_pair_keys = set(pair_metadata.loc[pair_metadata[COHORT_COLUMN].eq(cohort_year), "_pair_key"])
        if len(prompt_pair_keys) < 2:
            split_audit.append({
                "round_id": round_id,
                "cohort_year": int(cohort_year),
                "status": "skipped_one_prompt_pair",
            })
            continue

        validation_pair_key = prompt_pair_keys[-1]  # deterministic whole-pair validation split
        train_pair_keys.update(set(prompt_pair_keys) - {validation_pair_key})
        validation_pair_keys.add(validation_pair_key)
        test_pair_keys.update(all_pair_keys - set(prompt_pair_keys))
        usable_cohorts.append(int(cohort_year))

    if not usable_cohorts:
        continue

    assert train_pair_keys.isdisjoint(validation_pair_keys)
    assert train_pair_keys.isdisjoint(test_pair_keys)
    assert validation_pair_keys.isdisjoint(test_pair_keys)

    synthetic_train = round_synthetic.loc[round_synthetic["_pair_key"].isin(train_pair_keys)].copy()
    real_train = real_rows_for_pair_keys(train_pair_keys)
    real_validation = real_rows_for_pair_keys(validation_pair_keys)
    real_test = real_rows_for_pair_keys(test_pair_keys)

    real_train = real_train.loc[real_train[COHORT_COLUMN].isin(usable_cohorts)]
    real_validation = real_validation.loc[real_validation[COHORT_COLUMN].isin(usable_cohorts)]
    real_test = real_test.loc[real_test[COHORT_COLUMN].isin(usable_cohorts)]

    # Child-level separation within a training round.
    train_children = child_keys(real_train)
    validation_children = child_keys(real_validation)
    test_children = child_keys(real_test)
    assert train_children.isdisjoint(validation_children), f"Train/validation child overlap in {round_id}."
    assert train_children.isdisjoint(test_children), f"Train/test child overlap in {round_id}."
    assert validation_children.isdisjoint(test_children), f"Validation/test child overlap in {round_id}."

    usable_prompt_keys = train_pair_keys | validation_pair_keys
    prompt_children = child_keys(pair_children.loc[pair_children["_pair_key"].isin(usable_prompt_keys)])
    assert not test_children.intersection(prompt_children), f"Prompt/test child overlap in {round_id}."

    assert set(synthetic_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_train[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_validation[GROUP_COLUMN]) == set(COMPARISON_GROUPS)
    assert set(real_test[GROUP_COLUMN]) == set(COMPARISON_GROUPS)

    round_dir = CLASSIFICATION_DIR / round_id
    round_dir.mkdir(parents=True, exist_ok=True)
    frames = {
        "train": classifier_frame(real_train, round_id),
        "validation": classifier_frame(real_validation, round_id),
        "synthetic": classifier_frame(synthetic_train, round_id),
        "test": classifier_frame(real_test, round_id),
    }
    for split_name, frame in frames.items():
        frame.to_csv(round_dir / f"{split_name}.csv", index=False)
    (round_dir / "label_mapping.json").write_text(json.dumps(LABEL_MAPPING, indent=2) + "\n")

    prepared_rounds[round_id] = {"path": round_dir, **frames}
    split_audit.append({
        "round_id": round_id,
        "status": "prepared",
        "cohorts": usable_cohorts,
        "train_pair_keys": len(train_pair_keys),
        "validation_pair_keys": len(validation_pair_keys),
        "test_pair_keys": len(test_pair_keys),
        "real_train_rows": len(frames["train"]),
        "real_validation_rows": len(frames["validation"]),
        "synthetic_train_rows": len(frames["synthetic"]),
        "real_test_rows": len(frames["test"]),
        "train_validation_child_overlap": 0,
        "train_test_child_overlap": 0,
        "validation_test_child_overlap": 0,
        "test_prompt_child_overlap": 0,
        "pair_mapping_source": "generated_artifact_metadata",
        "split_key": "(cohort_year, pair_id)",
    })

split_audit_df = pd.DataFrame(split_audit)
split_audit_df.to_csv(CLASSIFICATION_DIR / "split_audit.csv", index=False)
display(split_audit_df)
assert prepared_rounds, "No usable round-specific classification datasets were prepared."


# Explicit class-balance audit. This is useful when interpreting any SLI/TD prediction
# asymmetry: if these counts are balanced, a preference for SLI is not caused by a
# simple excess of SLI training rows.
label_balance_rows = []
for round_id, prepared in prepared_rounds.items():
    for split_name in ("train", "validation", "synthetic", "test"):
        frame = prepared[split_name]
        counts = frame["Label"].value_counts().to_dict()
        label_balance_rows.append({
            "training_round": round_id,
            "split": split_name,
            "n_rows": int(len(frame)),
            "n_SLI": int(counts.get(LABEL_MAPPING["SLI"], 0)),
            "n_TD": int(counts.get(LABEL_MAPPING["TD"], 0)),
            "sli_fraction": float((frame["Label"] == LABEL_MAPPING["SLI"]).mean()),
        })

classification_label_balance_audit = pd.DataFrame(label_balance_rows)
classification_label_balance_audit.to_csv(
    CLASSIFICATION_DIR / "classification_label_balance_audit.csv", index=False
)
print("Classification label-balance audit")
display(classification_label_balance_audit)


In [ ]:
# Fine-tune and test SynthTextEval classifiers. Each prediction row records BOTH the
# evaluation dataset identifier and the training round/model that produced it.
if RUN_CLASSIFICATION:
    import torch
    from synthtexteval.eval.downstream.classify.train_classifier import (
        Arguments, Classifier, ModelArguments, TrainingArguments,
    )

    assert torch.cuda.is_available(), "Enable a Colab GPU before running classifier training."
    prediction_frames = []
    training_settings = ("real-only", "synthetic-train-only", "synthetic-train-augment")

    for round_id, prepared in prepared_rounds.items():
        round_dir = prepared["path"]
        for training_setting in training_settings:
            setting_dir = round_dir / training_setting
            model_dir = setting_dir / "model"
            setting_dir.mkdir(parents=True, exist_ok=True)

            train_args = TrainingArguments(
                output_dir=str(setting_dir / "trainer"),
                num_train_epochs=3,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir),
                path_to_model=str(model_dir),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                synthetic_usage=training_setting,
                is_train=True,
            )
            Classifier(Arguments(train=train_args, model=model_args)).finetune_model()

            prediction_path = setting_dir / "test_predictions.csv"
            test_args = TrainingArguments(
                output_dir=str(setting_dir / "tester"),
                per_device_eval_batch_size=8,
                save_strategy="no",
                report_to=[],
            )
            test_model_args = ModelArguments(
                model_name=CLASSIFIER_MODEL,
                path_to_dataset=str(round_dir / "test.csv"),
                path_to_model=str(model_dir),
                path_to_output_csv=str(prediction_path),
                path_to_aggregated_results=str(setting_dir / "test_metrics.csv"),
                n_labels=len(LABEL_MAPPING),
                problem_type="single_label_classification",
                text_field="output_text",
                label_field="Label",
                retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "pair_id"],
                is_test=True,
            )
            Classifier(Arguments(train=test_args, model=test_model_args)).test_model()
            prediction_frame = pd.read_csv(prediction_path)
            prediction_frame["training_setting"] = training_setting
            prediction_frame["training_round"] = round_id
            prediction_frame["evaluation_set"] = "internal_enni"
            prediction_frames.append(prediction_frame)

            if RUN_GILLAM_EXTERNAL_TEST:
                external_prediction_path = setting_dir / "external_gillam_predictions.csv"
                external_test_args = TrainingArguments(
                    output_dir=str(setting_dir / "external_gillam_tester"),
                    per_device_eval_batch_size=8,
                    save_strategy="no",
                    report_to=[],
                )
                external_model_args = ModelArguments(
                    model_name=CLASSIFIER_MODEL,
                    path_to_dataset=str(EXTERNAL_GILLAM_DIR / "test.csv"),
                    path_to_model=str(model_dir),
                    path_to_output_csv=str(external_prediction_path),
                    path_to_aggregated_results=str(setting_dir / "external_gillam_metrics.csv"),
                    n_labels=len(LABEL_MAPPING),
                    problem_type="single_label_classification",
                    text_field="output_text",
                    label_field="Label",
                    retain_columns=["cohort_year", "classification_round", "group", "source_child_id", "story_id", "source_dataset", "sex"],
                    is_test=True,
                )
                Classifier(Arguments(train=external_test_args, model=external_model_args)).test_model()
                external_prediction_frame = pd.read_csv(external_prediction_path)
                external_prediction_frame["training_setting"] = training_setting
                external_prediction_frame["training_round"] = round_id
                external_prediction_frame["evaluation_set"] = "external_gillam"
                prediction_frames.append(external_prediction_frame)

    classification_predictions = pd.concat(prediction_frames, ignore_index=True)
    classification_predictions.to_csv(CLASSIFICATION_DIR / "all_round_predictions.csv", index=False)
    display(
        classification_predictions
        .groupby(["evaluation_set", "training_setting", "training_round", "Ground"])
        .size().rename("n_predictions").reset_index()
    )
else:
    print("Datasets prepared. Set RUN_CLASSIFICATION = True in a GPU runtime to train and test the models.")


### Age × class recall

The final classification outputs report SLI recall and TD recall separately within each age cohort. Age is **not** supplied to BERT as a predictive feature; `cohort_year` is used only after prediction to stratify performance. Internal ENNI tables cover the age cohorts present in ENNI. External Gillam tables contain only age brackets actually represented in Gillam, so an age-4 external row is intentionally absent when Gillam has no 4-year-old participants. Child-level age × class recall is also reported after averaging story probabilities within each child.


In [ ]:
# Round-wise downstream utility and fairness audit.
# Do NOT pool repeated predictions from multiple trained round-models as if they were
# independent observations. Compute metrics within each round, then summarize across rounds.
#
# In addition to accuracy/F1, report balanced accuracy, class-specific recall, and the
# predicted SLI rate. These are essential for external sets such as Gillam when class
# prevalence is unequal.
if "classification_predictions" in globals():
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
    from synthtexteval.eval.fairness.metrics import analyze_group_fairness_performance

    SLI_ID = LABEL_MAPPING["SLI"]
    TD_ID = LABEL_MAPPING["TD"]

    def class_recall(y_true, y_pred, label_id):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mask = y_true == label_id
        if not mask.any():
            return np.nan
        return float((y_pred[mask] == label_id).mean())

    def diagnostic_metrics(frame):
        y_true = frame["Ground"].to_numpy()
        y_pred = frame["Predicted"].to_numpy()
        return {
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "f1_micro": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
            "sli_recall": class_recall(y_true, y_pred, SLI_ID),
            "td_recall": class_recall(y_true, y_pred, TD_ID),
            "actual_sli_rate": float((y_true == SLI_ID).mean()),
            "predicted_sli_rate": float((y_pred == SLI_ID).mean()),
        }

    def age_class_recall_table(predictions, evaluation_set):
        """Recall for each true class within each age cohort and trained round-model."""
        rows = []
        for (training_setting, training_round, cohort_year), frame in predictions.groupby(
            ["training_setting", "training_round", COHORT_COLUMN],
            sort=True,
            dropna=False,
        ):
            for class_label, class_id in LABEL_MAPPING.items():
                class_frame = frame.loc[frame["Ground"].eq(class_id)]
                n_true = int(len(class_frame))
                true_positive = int(class_frame["Predicted"].eq(class_id).sum()) if n_true else 0
                rows.append({
                    "evaluation_set": evaluation_set,
                    "training_setting": training_setting,
                    "training_round": training_round,
                    COHORT_COLUMN: cohort_year,
                    "class_label": class_label,
                    "class_id": int(class_id),
                    "n_true_class": n_true,
                    "true_positive": true_positive,
                    "recall": (true_positive / n_true) if n_true else np.nan,
                })
        return pd.DataFrame(rows)

    def summarize_age_class_recall(age_class_frame):
        """Mean ± SD across trained rounds; absent age/class cells stay absent/NaN."""
        summary = (
            age_class_frame
            .groupby(["training_setting", COHORT_COLUMN, "class_label"], as_index=False, dropna=False)
            .agg(
                n_rounds_with_class=("recall", lambda values: int(values.notna().sum())),
                n_true_class_per_round_mean=("n_true_class", "mean"),
                n_true_class_per_round_min=("n_true_class", "min"),
                n_true_class_per_round_max=("n_true_class", "max"),
                recall_mean=("recall", "mean"),
                recall_sd=("recall", "std"),
                recall_min=("recall", "min"),
                recall_max=("recall", "max"),
            )
        )

        # Compact final table: one row per setting × age, separate SLI/TD columns.
        mean_wide = summary.pivot(
            index=["training_setting", COHORT_COLUMN],
            columns="class_label",
            values="recall_mean",
        ).rename(columns=lambda label: f"{str(label).lower()}_recall_mean")

        sd_wide = summary.pivot(
            index=["training_setting", COHORT_COLUMN],
            columns="class_label",
            values="recall_sd",
        ).rename(columns=lambda label: f"{str(label).lower()}_recall_sd")

        n_wide = summary.pivot(
            index=["training_setting", COHORT_COLUMN],
            columns="class_label",
            values="n_true_class_per_round_mean",
        ).rename(columns=lambda label: f"{str(label).lower()}_n_per_round_mean")

        compact = pd.concat([mean_wide, sd_wide, n_wide], axis=1).reset_index()
        return summary, compact

    internal_predictions = classification_predictions.loc[
        classification_predictions["evaluation_set"].eq("internal_enni")
    ].copy()

    internal_overall_rows = []
    internal_subgroup_performance_parts = []
    internal_fairness_parts = []

    for (training_setting, training_round), prediction_frame in internal_predictions.groupby(
        ["training_setting", "training_round"], sort=True
    ):
        row = {
            "training_setting": training_setting,
            "training_round": training_round,
            "n_predictions": int(len(prediction_frame)),
            "n_unique_children": int(prediction_frame["source_child_id"].nunique()),
        }
        row.update(diagnostic_metrics(prediction_frame))
        internal_overall_rows.append(row)

        performance, fairness = analyze_group_fairness_performance(
            prediction_frame,
            problem_type="multiclass",
            num_classes=len(LABEL_MAPPING),
            subgroup_type=COHORT_COLUMN,
        )
        performance = performance.copy()
        fairness = fairness.copy()
        performance["training_setting"] = training_setting
        performance["training_round"] = training_round
        fairness["training_setting"] = training_setting
        fairness["training_round"] = training_round
        internal_subgroup_performance_parts.append(performance)
        internal_fairness_parts.append(fairness)

    internal_overall_round_metrics = pd.DataFrame(internal_overall_rows)
    internal_overall_round_metrics.to_csv(
        CLASSIFICATION_DIR / "internal_overall_round_metrics.csv", index=False
    )

    internal_overall_summary = (
        internal_overall_round_metrics
        .groupby("training_setting", as_index=False)
        .agg(
            n_rounds=("training_round", "nunique"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_sd=("accuracy", "std"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd=("balanced_accuracy", "std"),
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_sd=("f1_macro", "std"),
            f1_micro_mean=("f1_micro", "mean"),
            f1_micro_sd=("f1_micro", "std"),
            sli_recall_mean=("sli_recall", "mean"),
            sli_recall_sd=("sli_recall", "std"),
            td_recall_mean=("td_recall", "mean"),
            td_recall_sd=("td_recall", "std"),
            predicted_sli_rate_mean=("predicted_sli_rate", "mean"),
            predicted_sli_rate_sd=("predicted_sli_rate", "std"),
            test_predictions_per_round_mean=("n_predictions", "mean"),
            unique_children_per_round_mean=("n_unique_children", "mean"),
        )
    )
    internal_overall_summary.to_csv(
        CLASSIFICATION_DIR / "internal_overall_summary_across_rounds.csv", index=False
    )

    # Age × class recall: e.g. TD recall among 4-year-olds, SLI recall among 4-year-olds.
    # Age is used only for subgroup evaluation; it is not an input feature to BERT.
    internal_age_class_recall_by_round = age_class_recall_table(
        internal_predictions,
        evaluation_set="internal_enni",
    )
    internal_age_class_recall_by_round.to_csv(
        CLASSIFICATION_DIR / "internal_age_class_recall_by_round.csv", index=False
    )
    internal_age_class_recall_summary, internal_age_class_recall_final = summarize_age_class_recall(
        internal_age_class_recall_by_round
    )
    internal_age_class_recall_summary.to_csv(
        CLASSIFICATION_DIR / "internal_age_class_recall_summary_across_rounds.csv", index=False
    )
    internal_age_class_recall_final.to_csv(
        CLASSIFICATION_DIR / "internal_age_class_recall_final_table.csv", index=False
    )

    internal_subgroup_performance = pd.concat(
        internal_subgroup_performance_parts, ignore_index=True
    )
    internal_subgroup_performance.to_csv(
        CLASSIFICATION_DIR / "internal_cohort_performance_by_round.csv", index=False
    )

    perf_required = {"Accuracy", "Group Type", "Num of Samples", "f1_macro", "f1_micro"}
    if perf_required.issubset(internal_subgroup_performance.columns):
        internal_cohort_summary = (
            internal_subgroup_performance
            .groupby(["training_setting", "Group Type"], as_index=False)
            .agg(
                n_rounds=("training_round", "nunique"),
                accuracy_mean=("Accuracy", "mean"),
                accuracy_sd=("Accuracy", "std"),
                f1_macro_mean=("f1_macro", "mean"),
                f1_macro_sd=("f1_macro", "std"),
                f1_micro_mean=("f1_micro", "mean"),
                f1_micro_sd=("f1_micro", "std"),
                samples_per_round_mean=("Num of Samples", "mean"),
                samples_per_round_min=("Num of Samples", "min"),
                samples_per_round_max=("Num of Samples", "max"),
            )
        )
        internal_cohort_summary.to_csv(
            CLASSIFICATION_DIR / "internal_cohort_summary_across_rounds.csv", index=False
        )
    else:
        internal_cohort_summary = pd.DataFrame()

    internal_round_fairness = pd.concat(internal_fairness_parts, ignore_index=True)
    internal_round_fairness.to_csv(
        CLASSIFICATION_DIR / "internal_cohort_fairness_by_round.csv", index=False
    )

    fairness_tokens = ("fair", "equal", "odds", "difference", "gap", "ratio", "score")
    fairness_metric_columns = [
        col for col in internal_round_fairness.select_dtypes(include="number").columns
        if any(token in col.lower() for token in fairness_tokens)
    ]
    if fairness_metric_columns:
        fairness_summary = (
            internal_round_fairness
            .groupby("training_setting")[fairness_metric_columns]
            .agg(["mean", "std"])
        )
        fairness_summary.columns = [f"{metric}_{stat}" for metric, stat in fairness_summary.columns]
        fairness_summary = fairness_summary.reset_index()
        fairness_summary.to_csv(
            CLASSIFICATION_DIR / "internal_fairness_summary_across_rounds.csv", index=False
        )
    else:
        fairness_summary = pd.DataFrame()

    print("Internal overall metrics by round")
    display(internal_overall_round_metrics)
    print("Internal overall mean ± SD across trained rounds")
    display(internal_overall_summary)
    print("Age × class recall by round (true-class recall within each cohort)")
    display(internal_age_class_recall_by_round)
    print("Final age × class recall table: mean ± SD across trained rounds")
    display(internal_age_class_recall_final)
    print("Cohort performance by round")
    display(internal_subgroup_performance)
    if not internal_cohort_summary.empty:
        print("Cohort performance mean ± SD across trained rounds")
        display(internal_cohort_summary)
    print("Fairness output by round (not pooled)")
    display(internal_round_fairness)
    if not fairness_summary.empty:
        print("Fairness metric mean ± SD across trained rounds")
        display(fairness_summary)

    if RUN_GILLAM_EXTERNAL_TEST:
        external_predictions = classification_predictions.loc[
            classification_predictions["evaluation_set"].eq("external_gillam")
        ].copy()
        external_predictions.to_csv(
            CLASSIFICATION_DIR / "all_external_gillam_predictions.csv", index=False
        )

        external_round_rows = []
        for (training_setting, training_round), frame in external_predictions.groupby(
            ["training_setting", "training_round"], sort=True
        ):
            row = {
                "training_setting": training_setting,
                "training_round": training_round,
                "n_predictions": int(len(frame)),
                "n_unique_children": int(frame["source_child_id"].nunique()),
                "n_sli": int((frame["Ground"] == SLI_ID).sum()),
                "n_td": int((frame["Ground"] == TD_ID).sum()),
            }
            row.update(diagnostic_metrics(frame))
            external_round_rows.append(row)

        external_round_metrics = pd.DataFrame(external_round_rows)
        external_round_metrics.to_csv(
            CLASSIFICATION_DIR / "external_gillam_round_metrics.csv", index=False
        )

        external_summary = (
            external_round_metrics
            .groupby("training_setting", as_index=False)
            .agg(
                n_rounds=("training_round", "nunique"),
                accuracy_mean=("accuracy", "mean"),
                accuracy_sd=("accuracy", "std"),
                balanced_accuracy_mean=("balanced_accuracy", "mean"),
                balanced_accuracy_sd=("balanced_accuracy", "std"),
                f1_macro_mean=("f1_macro", "mean"),
                f1_macro_sd=("f1_macro", "std"),
                f1_micro_mean=("f1_micro", "mean"),
                f1_micro_sd=("f1_micro", "std"),
                sli_recall_mean=("sli_recall", "mean"),
                sli_recall_sd=("sli_recall", "std"),
                td_recall_mean=("td_recall", "mean"),
                td_recall_sd=("td_recall", "std"),
                predicted_sli_rate_mean=("predicted_sli_rate", "mean"),
                predicted_sli_rate_sd=("predicted_sli_rate", "std"),
                unique_children=("n_unique_children", "max"),
                external_sli_children=("n_sli", "max"),
                external_td_children=("n_td", "max"),
            )
        )
        external_summary.to_csv(
            CLASSIFICATION_DIR / "external_gillam_summary_across_rounds.csv", index=False
        )

        # External age × class recall. Only cohorts actually present in Gillam appear.
        # In particular, no age-4 external row is created if Gillam has no 4-year-olds.
        external_age_class_recall_by_round = age_class_recall_table(
            external_predictions,
            evaluation_set="external_gillam",
        )
        external_age_class_recall_by_round.to_csv(
            CLASSIFICATION_DIR / "external_gillam_age_class_recall_by_round.csv", index=False
        )
        external_age_class_recall_summary, external_age_class_recall_final = summarize_age_class_recall(
            external_age_class_recall_by_round
        )
        external_age_class_recall_summary.to_csv(
            CLASSIFICATION_DIR / "external_gillam_age_class_recall_summary_across_rounds.csv", index=False
        )
        external_age_class_recall_final.to_csv(
            CLASSIFICATION_DIR / "external_gillam_age_class_recall_final_table.csv", index=False
        )

        # Prevalence-independent context for interpreting accuracy.
        gillam_prevalence_audit = {
            "n": int(len(external_gillam_test)),
            "n_sli": int((external_gillam_test["Label"] == SLI_ID).sum()),
            "n_td": int((external_gillam_test["Label"] == TD_ID).sum()),
            "sli_prevalence": float((external_gillam_test["Label"] == SLI_ID).mean()),
            "majority_class_accuracy_if_always_predict_sli": float(
                (external_gillam_test["Label"] == SLI_ID).mean()
            ),
            "note": (
                "Adding more TD cases improves external precision and reduces prevalence-driven "
                "accuracy distortion, but Gillam is test-only and therefore cannot change the "
                "trained classifier's decision boundary."
            ),
        }
        (CLASSIFICATION_DIR / "external_gillam_prevalence_audit.json").write_text(
            json.dumps(gillam_prevalence_audit, indent=2) + "\n"
        )

        print("Gillam external metrics by trained round-model")
        display(external_round_metrics)
        print("Gillam external mean ± SD across trained round-models")
        display(external_summary)
        print("Gillam age × class recall by round")
        display(external_age_class_recall_by_round)
        print("Gillam final age × class recall table (only observed age brackets)")
        display(external_age_class_recall_final)
        print("Gillam prevalence audit")
        print(gillam_prevalence_audit)
else:
    print("Run the classifier cell first; no prediction file has been created.")


### Child-level downstream classification

Internal ENNI rows are story-level, while TD/SLI is a child-level label. This supplementary analysis loads each trained round-model, obtains class probabilities for every held-out story, averages probabilities across all stories from the same child, and makes exactly one prediction per child. Metrics are computed within round and then summarized across rounds.


In [ ]:
if RUN_CHILD_LEVEL_CLASSIFICATION and RUN_CLASSIFICATION:
    import gc
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

    CHILD_LEVEL_DIR = CLASSIFICATION_DIR / "child_level"
    CHILD_LEVEL_DIR.mkdir(parents=True, exist_ok=True)

    SLI_ID = LABEL_MAPPING["SLI"]
    TD_ID = LABEL_MAPPING["TD"]
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CHILD_BATCH_SIZE = 32
    CHILD_MAX_LENGTH = 512

    def _class_recall(y_true, y_pred, label_id):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mask = y_true == label_id
        return float((y_pred[mask] == label_id).mean()) if mask.any() else np.nan

    def _load_classifier(round_id, training_setting):
        model_dir = prepared_rounds[round_id]["path"] / training_setting / "model"
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        except Exception:
            tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_dir, local_files_only=True
        ).to(DEVICE)
        model.eval()
        return tokenizer, model

    story_probability_parts = []
    child_prediction_parts = []
    child_round_rows = []

    for round_id, prepared in prepared_rounds.items():
        test_frame = prepared["test"].copy().reset_index(drop=True)

        for training_setting in (
            "real-only", "synthetic-train-only", "synthetic-train-augment"
        ):
            tokenizer, model = _load_classifier(round_id, training_setting)
            probability_chunks = []

            for start in range(0, len(test_frame), CHILD_BATCH_SIZE):
                batch = test_frame.iloc[start:start + CHILD_BATCH_SIZE]
                encoded = tokenizer(
                    batch["output_text"].astype(str).tolist(),
                    padding=True,
                    truncation=True,
                    max_length=CHILD_MAX_LENGTH,
                    return_tensors="pt",
                )
                encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
                with torch.no_grad():
                    logits = model(**encoded).logits
                    probs = torch.softmax(logits, dim=-1).cpu().numpy()
                probability_chunks.append(probs)

            probabilities = np.vstack(probability_chunks)
            story_probs = test_frame[
                ["source_child_id", "cohort_year", "group", "pair_id", "Label", "output_text"]
            ].copy()
            story_probs["training_round"] = round_id
            story_probs["training_setting"] = training_setting
            story_probs["p_SLI"] = probabilities[:, SLI_ID]
            story_probs["p_TD"] = probabilities[:, TD_ID]
            story_probs["story_predicted"] = probabilities.argmax(axis=1)
            story_probability_parts.append(story_probs)

            child_probs = (
                story_probs
                .groupby(
                    [
                        "source_child_id", "cohort_year", "group", "pair_id",
                        "Label", "training_round", "training_setting"
                    ],
                    as_index=False,
                )
                .agg(
                    n_stories=("output_text", "size"),
                    mean_p_SLI=("p_SLI", "mean"),
                    mean_p_TD=("p_TD", "mean"),
                )
            )
            child_probs["Predicted"] = np.where(
                child_probs["mean_p_SLI"] >= child_probs["mean_p_TD"],
                SLI_ID,
                TD_ID,
            )
            child_probs["Ground"] = child_probs["Label"].astype(int)
            child_probs["correct"] = child_probs["Predicted"].eq(child_probs["Ground"])
            child_prediction_parts.append(child_probs)

            y_true = child_probs["Ground"].to_numpy()
            y_pred = child_probs["Predicted"].to_numpy()
            child_round_rows.append({
                "training_setting": training_setting,
                "training_round": round_id,
                "n_children": int(len(child_probs)),
                "accuracy": float(accuracy_score(y_true, y_pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
                "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
                "sli_recall": _class_recall(y_true, y_pred, SLI_ID),
                "td_recall": _class_recall(y_true, y_pred, TD_ID),
                "predicted_sli_rate": float((y_pred == SLI_ID).mean()),
            })

            del model, tokenizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    child_story_probabilities = pd.concat(story_probability_parts, ignore_index=True)
    child_level_predictions = pd.concat(child_prediction_parts, ignore_index=True)
    child_level_round_metrics = pd.DataFrame(child_round_rows)

    child_story_probabilities.to_csv(
        CHILD_LEVEL_DIR / "internal_story_probabilities.csv", index=False
    )
    child_level_predictions.to_csv(
        CHILD_LEVEL_DIR / "internal_child_predictions.csv", index=False
    )
    child_level_round_metrics.to_csv(
        CHILD_LEVEL_DIR / "internal_child_metrics_by_round.csv", index=False
    )

    child_level_summary = (
        child_level_round_metrics
        .groupby("training_setting", as_index=False)
        .agg(
            n_rounds=("training_round", "nunique"),
            children_per_round_mean=("n_children", "mean"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_sd=("accuracy", "std"),
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd=("balanced_accuracy", "std"),
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_sd=("f1_macro", "std"),
            sli_recall_mean=("sli_recall", "mean"),
            sli_recall_sd=("sli_recall", "std"),
            td_recall_mean=("td_recall", "mean"),
            td_recall_sd=("td_recall", "std"),
            predicted_sli_rate_mean=("predicted_sli_rate", "mean"),
            predicted_sli_rate_sd=("predicted_sli_rate", "std"),
        )
    )
    child_level_summary.to_csv(
        CHILD_LEVEL_DIR / "internal_child_summary_across_rounds.csv", index=False
    )

    # Child-level age × class recall after averaging story probabilities within each child.
    child_age_class_rows = []
    for (training_setting, training_round, cohort_year), frame in child_level_predictions.groupby(
        ["training_setting", "training_round", "cohort_year"],
        sort=True,
        dropna=False,
    ):
        for class_label, class_id in LABEL_MAPPING.items():
            class_frame = frame.loc[frame["Ground"].eq(class_id)]
            n_true = int(len(class_frame))
            true_positive = int(class_frame["Predicted"].eq(class_id).sum()) if n_true else 0
            child_age_class_rows.append({
                "training_setting": training_setting,
                "training_round": training_round,
                "cohort_year": cohort_year,
                "class_label": class_label,
                "class_id": int(class_id),
                "n_true_children": n_true,
                "true_positive_children": true_positive,
                "recall": (true_positive / n_true) if n_true else np.nan,
            })

    child_age_class_recall_by_round = pd.DataFrame(child_age_class_rows)
    child_age_class_recall_by_round.to_csv(
        CHILD_LEVEL_DIR / "internal_child_age_class_recall_by_round.csv", index=False
    )

    child_age_class_recall_summary = (
        child_age_class_recall_by_round
        .groupby(["training_setting", "cohort_year", "class_label"], as_index=False, dropna=False)
        .agg(
            n_rounds_with_class=("recall", lambda values: int(values.notna().sum())),
            n_true_children_per_round_mean=("n_true_children", "mean"),
            recall_mean=("recall", "mean"),
            recall_sd=("recall", "std"),
            recall_min=("recall", "min"),
            recall_max=("recall", "max"),
        )
    )
    child_age_class_recall_summary.to_csv(
        CHILD_LEVEL_DIR / "internal_child_age_class_recall_summary_across_rounds.csv", index=False
    )

    child_mean_wide = child_age_class_recall_summary.pivot(
        index=["training_setting", "cohort_year"],
        columns="class_label",
        values="recall_mean",
    ).rename(columns=lambda label: f"{str(label).lower()}_recall_mean")
    child_sd_wide = child_age_class_recall_summary.pivot(
        index=["training_setting", "cohort_year"],
        columns="class_label",
        values="recall_sd",
    ).rename(columns=lambda label: f"{str(label).lower()}_recall_sd")
    child_age_class_recall_final = pd.concat([child_mean_wide, child_sd_wide], axis=1).reset_index()
    child_age_class_recall_final.to_csv(
        CHILD_LEVEL_DIR / "internal_child_age_class_recall_final_table.csv", index=False
    )

    print("Child-level metrics by trained round-model")
    display(child_level_round_metrics)
    print("Child-level mean ± SD across rounds")
    display(child_level_summary)
    print("Child-level age × class recall by round")
    display(child_age_class_recall_by_round)
    print("Child-level final age × class recall table")
    display(child_age_class_recall_final)
else:
    print("Child-level classification skipped.")


In [ ]:
%pip install -q captum

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from captum.attr import LayerIntegratedGradients

EXPLAIN_DIR = CLASSIFICATION_DIR / "explainability"
EXPLAIN_DIR.mkdir(parents=True, exist_ok=True)

# Start reasonably small. Integrated Gradients is much more expensive
# than ordinary prediction.
EXPLAIN_SAMPLES_PER_CLASS = 10

# Adaptive integration: start at 32 steps and increase only for explanations whose
# Captum completeness/convergence delta is still too large.
EXPLAIN_N_STEPS_SEQUENCE = (32, 64, 128)
EXPLAIN_CONVERGENCE_ABS_TOL = 0.10

EXPLAIN_INTERNAL_BATCH_SIZE = 8
EXPLAIN_MAX_LENGTH = 512
EXPLAIN_SEED = 42

TRAINING_SETTINGS = (
    "real-only",
    "synthetic-train-only",
    "synthetic-train-augment",
)

SLI_ID = LABEL_MAPPING["SLI"]  # 0
TD_ID = LABEL_MAPPING["TD"]    # 1

ID_TO_LABEL = {
    value: key
    for key, value in LABEL_MAPPING.items()
}

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Explainability output:", EXPLAIN_DIR)
print("Device:", DEVICE)
print("SLI class:", SLI_ID)
print("TD class:", TD_ID)

explainability_config = {
    "method": "Layer Integrated Gradients",
    "explained_quantity": "SLI logit - TD logit",
    "positive_attribution": "evidence toward SLI",
    "negative_attribution": "evidence toward TD",
    "samples_per_class_per_round": EXPLAIN_SAMPLES_PER_CLASS,
    "n_steps_sequence": list(EXPLAIN_N_STEPS_SEQUENCE),
    "convergence_abs_tolerance": EXPLAIN_CONVERGENCE_ABS_TOL,
    "internal_batch_size": EXPLAIN_INTERNAL_BATCH_SIZE,
    "max_length": EXPLAIN_MAX_LENGTH,
    "seed": EXPLAIN_SEED,
    "training_settings": list(TRAINING_SETTINGS),
    "label_mapping": LABEL_MAPPING,
}

(EXPLAIN_DIR / "explainability_config.json").write_text(
    json.dumps(explainability_config, indent=2) + "\n"
)

In [ ]:
sample_parts = []

for round_index, (round_id, prepared) in enumerate(
    prepared_rounds.items()
):
    test_frame = prepared["test"].copy()

    for label_id in sorted(LABEL_MAPPING.values()):
        class_frame = test_frame.loc[
            test_frame["Label"].eq(label_id)
        ].copy()

        if EXPLAIN_SAMPLES_PER_CLASS is None:
            sampled = class_frame
        else:
            n = min(
                EXPLAIN_SAMPLES_PER_CLASS,
                len(class_frame),
            )

            sampled = class_frame.sample(
                n=n,
                random_state=(
                    EXPLAIN_SEED
                    + 100 * round_index
                    + label_id
                ),
            )

        sampled = sampled.copy()
        sampled["training_round"] = round_id
        sample_parts.append(sampled)

explainability_samples = pd.concat(
    sample_parts,
    ignore_index=True,
)

# Stable identifier for explanation outputs
explainability_samples["explanation_id"] = [
    f"{row.training_round}_child_{row.source_child_id}"
    for row in explainability_samples.itertuples()
]

explainability_samples.to_csv(
    EXPLAIN_DIR / "explainability_sample_manifest.csv",
    index=False,
)

print("Examples selected:", len(explainability_samples))

display(
    explainability_samples
    .groupby(["training_round", "group"])
    .size()
    .rename("n")
    .reset_index()
)

In [ ]:
def load_explanation_model(round_id, training_setting):
    model_dir = (
        prepared_rounds[round_id]["path"]
        / training_setting
        / "model"
    )

    if not model_dir.exists():
        raise FileNotFoundError(
            f"Missing trained model: {model_dir}"
        )

    # Some SynthTextEval saves may not contain tokenizer files,
    # so fall back to the original base tokenizer.
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            local_files_only=True,
        )
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(
            CLASSIFIER_MODEL
        )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            model_dir,
            local_files_only=True,
        )
        .to(DEVICE)
    )

    model.eval()

    return tokenizer, model, model_dir


def build_explainer(model):
    # Return a scalar margin for each example:
    #
    #     positive -> SLI
    #     negative -> TD
    #
    def forward_margin(
        input_ids,
        attention_mask,
        token_type_ids,
    ):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        logits = outputs.logits

        return (
            logits[:, SLI_ID]
            - logits[:, TD_ID]
        )

    return LayerIntegratedGradients(
        forward_margin,
        model.get_input_embeddings(),
    )


def explain_text(
    text,
    tokenizer,
    model,
    explainer,
):
    # Count original tokens so we can record whether the classifier
    # had to truncate this transcript.
    original_ids = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]
    original_token_count = len(original_ids)

    encoded = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        max_length=EXPLAIN_MAX_LENGTH,
        padding="max_length",
        return_attention_mask=True,
        return_special_tokens_mask=True,
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)
    token_type_ids = encoded.get(
        "token_type_ids",
        torch.zeros_like(input_ids),
    ).to(DEVICE)

    special_tokens_mask = (
        encoded["special_tokens_mask"]
        .squeeze(0)
        .cpu()
        .numpy()
    )

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predicted_id = int(probabilities.argmax(dim=-1).item())
        sli_probability = float(probabilities[0, SLI_ID].item())
        td_probability = float(probabilities[0, TD_ID].item())
        sli_td_margin = float(
            (logits[0, SLI_ID] - logits[0, TD_ID]).item()
        )

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    baseline_ids = torch.full_like(input_ids, fill_value=pad_id)

    # Preserve special tokens in the baseline while replacing content with padding.
    for token_index, is_special in enumerate(special_tokens_mask):
        if is_special:
            baseline_ids[0, token_index] = input_ids[0, token_index]

    attributions = None
    convergence_delta = None
    n_steps_used = None

    # Increase integration resolution only when needed.
    for n_steps in EXPLAIN_N_STEPS_SEQUENCE:
        attributions, convergence_delta = explainer.attribute(
            inputs=input_ids,
            baselines=baseline_ids,
            additional_forward_args=(attention_mask, token_type_ids),
            n_steps=n_steps,
            internal_batch_size=EXPLAIN_INTERNAL_BATCH_SIZE,
            return_convergence_delta=True,
        )
        n_steps_used = int(n_steps)
        delta_value = float(
            convergence_delta.detach().cpu().flatten()[0].item()
        )
        if abs(delta_value) <= EXPLAIN_CONVERGENCE_ABS_TOL:
            break

    token_attr = (
        attributions
        .sum(dim=-1)
        .squeeze(0)
        .detach()
        .cpu()
    )

    input_ids_cpu = input_ids.squeeze(0).detach().cpu()
    attention_cpu = attention_mask.squeeze(0).detach().cpu()
    tokens = tokenizer.convert_ids_to_tokens(input_ids_cpu.tolist())

    valid_indices = [
        i
        for i in range(len(tokens))
        if attention_cpu[i].item() == 1
        and special_tokens_mask[i] == 0
    ]

    raw_scores = np.array(
        [float(token_attr[i].item()) for i in valid_indices],
        dtype=float,
    )

    denominator = np.abs(raw_scores).sum()
    normalized_scores = raw_scores / denominator if denominator > 0 else raw_scores

    token_frame = pd.DataFrame({
        "token_index": valid_indices,
        "token": [tokens[i] for i in valid_indices],
        "attribution_raw": raw_scores,
        "attribution_normalized": normalized_scores,
    })

    delta_value = float(
        convergence_delta.detach().cpu().flatten()[0].item()
    )
    summary = {
        "predicted_id": predicted_id,
        "predicted_label": ID_TO_LABEL[predicted_id],
        "p_SLI": sli_probability,
        "p_TD": td_probability,
        "sli_minus_td_logit_margin": sli_td_margin,
        "original_token_count": original_token_count,
        "used_token_count": int(attention_cpu.sum().item()),
        "was_truncated": original_token_count > EXPLAIN_MAX_LENGTH,
        "n_steps_used": n_steps_used,
        "convergence_delta": delta_value,
        "abs_convergence_delta": abs(delta_value),
        "convergence_within_tolerance": (
            abs(delta_value) <= EXPLAIN_CONVERGENCE_ABS_TOL
        ),
    }

    return token_frame, summary


In [ ]:
from tqdm.auto import tqdm
import gc

all_token_parts = []
all_example_parts = []

for round_id in prepared_rounds:

    round_samples = (
        explainability_samples.loc[
            explainability_samples[
                "training_round"
            ].eq(round_id)
        ]
        .copy()
        .reset_index(drop=True)
    )

    for training_setting in TRAINING_SETTINGS:

        print(
            "\n",
            "=" * 80,
            f"\n{round_id} | {training_setting}\n",
            "=" * 80,
        )

        tokenizer, model, model_dir = (
            load_explanation_model(
                round_id,
                training_setting,
            )
        )

        explainer = build_explainer(model)

        model_token_parts = []
        model_example_rows = []

        for row in tqdm(
            round_samples.itertuples(index=False),
            total=len(round_samples),
        ):
            token_frame, summary = explain_text(
                text=row.output_text,
                tokenizer=tokenizer,
                model=model,
                explainer=explainer,
            )

            token_frame["explanation_id"] = (
                row.explanation_id
            )
            token_frame["training_round"] = round_id
            token_frame["training_setting"] = (
                training_setting
            )
            token_frame["source_child_id"] = (
                row.source_child_id
            )
            token_frame["cohort_year"] = (
                row.cohort_year
            )
            token_frame["ground_id"] = row.Label
            token_frame["ground_label"] = (
                ID_TO_LABEL[int(row.Label)]
            )

            model_token_parts.append(token_frame)

            model_example_rows.append({
                "explanation_id": row.explanation_id,
                "training_round": round_id,
                "training_setting": training_setting,
                "source_child_id": row.source_child_id,
                "pair_id": row.pair_id,
                "cohort_year": row.cohort_year,
                "ground_id": int(row.Label),
                "ground_label": (
                    ID_TO_LABEL[int(row.Label)]
                ),
                "predicted_id": summary[
                    "predicted_id"
                ],
                "predicted_label": summary[
                    "predicted_label"
                ],
                "correct": (
                    int(row.Label)
                    == summary["predicted_id"]
                ),
                "p_SLI": summary["p_SLI"],
                "p_TD": summary["p_TD"],
                "sli_minus_td_logit_margin": (
                    summary[
                        "sli_minus_td_logit_margin"
                    ]
                ),
                "original_token_count": (
                    summary[
                        "original_token_count"
                    ]
                ),
                "used_token_count": (
                    summary["used_token_count"]
                ),
                "was_truncated": (
                    summary["was_truncated"]
                ),
                "n_steps_used": summary["n_steps_used"],
                "convergence_delta": summary["convergence_delta"],
                "abs_convergence_delta": summary["abs_convergence_delta"],
                "convergence_within_tolerance": summary["convergence_within_tolerance"],
                "model_dir": str(model_dir),
                "text": row.output_text,
            })

        model_tokens = pd.concat(
            model_token_parts,
            ignore_index=True,
        )

        model_examples = pd.DataFrame(
            model_example_rows
        )

        safe_setting = training_setting.replace(
            "-",
            "_",
        )

        # Save after every model/round.
        model_tokens.to_csv(
            EXPLAIN_DIR
            / f"{round_id}_{safe_setting}_token_attributions.csv",
            index=False,
        )

        model_examples.to_csv(
            EXPLAIN_DIR
            / f"{round_id}_{safe_setting}_example_predictions.csv",
            index=False,
        )

        all_token_parts.append(model_tokens)
        all_example_parts.append(model_examples)

        del explainer
        del model
        del tokenizer

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


all_token_attributions = pd.concat(
    all_token_parts,
    ignore_index=True,
)

all_example_explanations = pd.concat(
    all_example_parts,
    ignore_index=True,
)

all_token_attributions.to_csv(
    EXPLAIN_DIR / "all_token_attributions.csv",
    index=False,
)

all_example_explanations.to_csv(
    EXPLAIN_DIR / "all_example_predictions.csv",
    index=False,
)

print("Explainability run complete.")
print("Token rows:", len(all_token_attributions))
print("Example/model combinations:", len(all_example_explanations))

In [ ]:
def merge_wordpieces(frame):
    rows = []

    for (
        explanation_id,
        training_round,
        training_setting,
    ), group in frame.groupby(
        [
            "explanation_id",
            "training_round",
            "training_setting",
        ],
        sort=False,
    ):

        group = group.sort_values(
            "token_index"
        )

        current = None

        for row in group.itertuples():

            token = str(row.token)

            if (
                token.startswith("##")
                and current is not None
            ):
                current["word"] += token[2:]
                current["attribution_raw"] += (
                    row.attribution_raw
                )
                current[
                    "attribution_normalized"
                ] += row.attribution_normalized

            else:
                if current is not None:
                    rows.append(current)

                current = {
                    "explanation_id": explanation_id,
                    "training_round": training_round,
                    "training_setting": training_setting,
                    "source_child_id": (
                        row.source_child_id
                    ),
                    "cohort_year": row.cohort_year,
                    "ground_id": row.ground_id,
                    "ground_label": (
                        row.ground_label
                    ),
                    "word": token,
                    "attribution_raw": (
                        row.attribution_raw
                    ),
                    "attribution_normalized": (
                        row.attribution_normalized
                    ),
                }

        if current is not None:
            rows.append(current)

    return pd.DataFrame(rows)


word_attributions = merge_wordpieces(
    all_token_attributions
)

# Normalize textual representation for aggregation.
word_attributions["word"] = (
    word_attributions["word"]
    .astype(str)
    .str.lower()
)

word_attributions.to_csv(
    EXPLAIN_DIR / "all_word_attributions.csv",
    index=False,
)


# Collapse repeated occurrences of the same word
# within one transcript first.
document_word_attributions = (
    word_attributions
    .groupby(
        [
            "training_round",
            "training_setting",
            "explanation_id",
            "source_child_id",
            "word",
        ],
        as_index=False,
    )
    .agg(
        signed_attribution=(
            "attribution_normalized",
            "sum",
        ),
        absolute_attribution=(
            "attribution_normalized",
            lambda values: np.abs(values).sum(),
        ),
    )
)

document_word_attributions.to_csv(
    EXPLAIN_DIR
    / "document_word_attributions.csv",
    index=False,
)


# Number of explained documents for every model/round.
n_documents = (
    all_example_explanations
    .groupby(
        [
            "training_round",
            "training_setting",
        ]
    )["explanation_id"]
    .nunique()
    .rename("n_explained_documents")
    .reset_index()
)


# Aggregate word importance INSIDE each trained round-model.
word_by_round = (
    document_word_attributions
    .groupby(
        [
            "training_round",
            "training_setting",
            "word",
        ],
        as_index=False,
    )
    .agg(
        document_frequency=(
            "explanation_id",
            "nunique",
        ),
        total_signed_attribution=(
            "signed_attribution",
            "sum",
        ),
        total_absolute_attribution=(
            "absolute_attribution",
            "sum",
        ),
    )
)

word_by_round = word_by_round.merge(
    n_documents,
    on=["training_round", "training_setting"],
    how="left",
)

# Divide by ALL explained documents, not only documents
# containing the word. This incorporates both importance
# and prevalence.
word_by_round[
    "signed_importance_per_document"
] = (
    word_by_round["total_signed_attribution"]
    / word_by_round["n_explained_documents"]
)

word_by_round[
    "absolute_importance_per_document"
] = (
    word_by_round["total_absolute_attribution"]
    / word_by_round["n_explained_documents"]
)

word_by_round[
    "document_prevalence"
] = (
    word_by_round["document_frequency"]
    / word_by_round["n_explained_documents"]
)

word_by_round.to_csv(
    EXPLAIN_DIR
    / "word_importance_by_round.csv",
    index=False,
)


# Now summarize ACROSS rounds.
word_across_rounds = (
    word_by_round
    .groupby(
        [
            "training_setting",
            "word",
        ],
        as_index=False,
    )
    .agg(
        n_rounds=("training_round", "nunique"),
        signed_importance_mean=(
            "signed_importance_per_document",
            "mean",
        ),
        signed_importance_sd=(
            "signed_importance_per_document",
            "std",
        ),
        absolute_importance_mean=(
            "absolute_importance_per_document",
            "mean",
        ),
        absolute_importance_sd=(
            "absolute_importance_per_document",
            "std",
        ),
        document_prevalence_mean=(
            "document_prevalence",
            "mean",
        ),
    )
)

word_across_rounds.to_csv(
    EXPLAIN_DIR
    / "word_importance_summary_across_rounds.csv",
    index=False,
)


# Top globally influential words for each training condition.
top_words = (
    word_across_rounds
    .sort_values(
        [
            "training_setting",
            "absolute_importance_mean",
        ],
        ascending=[True, False],
    )
    .groupby(
        "training_setting",
        group_keys=False,
    )
    .head(30)
    .reset_index(drop=True)
)

top_words.to_csv(
    EXPLAIN_DIR
    / "top_30_words_by_training_setting.csv",
    index=False,
)

display(top_words)

In [ ]:
# Side-by-side predictions for the SAME transcript
prediction_comparison = (
    all_example_explanations
    .pivot_table(
        index=[
            "training_round",
            "explanation_id",
            "source_child_id",
            "cohort_year",
            "ground_label",
        ],
        columns="training_setting",
        values=[
            "predicted_label",
            "p_SLI",
            "p_TD",
            "sli_minus_td_logit_margin",
        ],
        aggfunc="first",
    )
)

prediction_comparison.columns = [
    f"{metric}__{setting}"
    for metric, setting
    in prediction_comparison.columns
]

prediction_comparison = (
    prediction_comparison
    .reset_index()
)

prediction_comparison.to_csv(
    EXPLAIN_DIR
    / "same_text_model_comparison.csv",
    index=False,
)


# Integrated Gradients convergence diagnostics
ig_diagnostics = (
    all_example_explanations
    .groupby(
        [
            "training_setting",
            "training_round",
        ],
        as_index=False,
    )
    .agg(
        n_explanations=(
            "explanation_id",
            "size",
        ),
        mean_abs_convergence_delta=(
            "convergence_delta",
            lambda x: np.abs(x).mean(),
        ),
        median_abs_convergence_delta=(
            "convergence_delta",
            lambda x: np.median(np.abs(x)),
        ),
        max_abs_convergence_delta=(
            "convergence_delta",
            lambda x: np.abs(x).max(),
        ),
        mean_n_steps_used=(
            "n_steps_used",
            "mean",
        ),
        max_n_steps_used=(
            "n_steps_used",
            "max",
        ),
        fraction_within_convergence_tolerance=(
            "convergence_within_tolerance",
            "mean",
        ),
        truncation_rate=(
            "was_truncated",
            "mean",
        ),
    )
)

ig_diagnostics.to_csv(
    EXPLAIN_DIR
    / "integrated_gradients_diagnostics.csv",
    index=False,
)

display(prediction_comparison.head())
display(ig_diagnostics)

print(
    "\nSaved explainability results to:",
    EXPLAIN_DIR,
)

## Privacy and memorization

This section restores SynthTextEval's package-native `entity_leakage` and `search_and_compute_EPO` analyses. The package expects a user-supplied list of private entities, but this project has no such list because the stories are de-identified. We therefore extract candidate named entities from the balanced real corpus and report their occurrence and surrounding-span overlap in the synthetic corpus. These are reference-overlap diagnostics—not verified private-entity leakage rates. Shared ENNI character names, places, and NER errors can produce matches.

Canary attacks are not applicable retrospectively: they require known secrets to have been deliberately inserted during generator training. No canaries were planted in the model used to produce these narratives.

In [ ]:
if RUN_PRIVACY_ANALYSIS:
    from collections import Counter, defaultdict
    import spacy
    from synthtexteval.eval.privacy.metrics import entity_leakage, search_and_compute_EPO

    PRIVACY_DIR = RESULTS_DIR / "privacy"
    PRIVACY_DIR.mkdir(parents=True, exist_ok=True)
    # Privacy/reference-overlap analysis should use the full available corpora, not the
    # balanced descriptive subsample. If the generator was trained/prompted from a narrower
    # known source corpus, replace real_privacy_texts with that exact source set.
    synthetic_privacy_texts = (
        analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("synthetic"), TEXT_COLUMN]
        .dropna().astype(str).tolist()
    )
    real_privacy_texts = (
        analysis_records.loc[analysis_records[SOURCE_COLUMN].eq("real"), TEXT_COLUMN]
        .dropna().astype(str).tolist()
    )

    # The toolkit requires private entities supplied by the user. No protected-entity list
    # exists here, so use NER candidates from real ENNI and retain their provenance.
    nlp = spacy.load("en_core_web_sm")
    entity_document_frequency = Counter()
    entity_labels = defaultdict(Counter)
    entity_surface = {}
    for document in nlp.pipe(real_privacy_texts, batch_size=64):
        seen_in_document = set()
        for entity in document.ents:
            surface = " ".join(entity.text.split()).strip()
            if entity.label_ not in PRIVACY_ENTITY_LABELS or len(surface) < 2 or surface.isdigit():
                continue
            key = surface.casefold()
            entity_surface.setdefault(key, surface)
            entity_labels[key][entity.label_] += 1
            if key not in seen_in_document:
                entity_document_frequency[key] += 1
                seen_in_document.add(key)

    entity_candidates = [entity_surface[key] for key in sorted(entity_surface)]
    candidate_rows = []
    for key in sorted(entity_surface):
        candidate_rows.append({
            "entity": entity_surface[key],
            "ner_labels": "|".join(sorted(entity_labels[key])),
            "real_document_frequency": int(entity_document_frequency[key]),
            "verified_private_entity": False,
        })
    pd.DataFrame(candidate_rows, columns=[
        "entity", "ner_labels", "real_document_frequency", "verified_private_entity"
    ]).to_csv(PRIVACY_DIR / "reference_named_entity_candidates.csv", index=False)

    if entity_candidates:
        average_entity_matches, entity_details = entity_leakage(
            paragraphs=synthetic_privacy_texts,
            entities=entity_candidates,
            entity_leakage_result_path=str(PRIVACY_DIR / "entity_leakage.pkl"),
        )
        leakage_rows = []
        leaked_entity_union = set()
        for row_index, text in enumerate(synthetic_privacy_texts):
            entity_flags, match_count = entity_details[text]
            matched_entities = sorted(entity for entity, found in entity_flags.items() if found)
            leaked_entity_union.update(matched_entities)
            leakage_rows.append({
                "comparison_row": row_index,
                "candidate_entity_match_count": int(match_count),
                "matched_candidate_entities": "|".join(matched_entities),
            })
        pd.DataFrame(leakage_rows).to_csv(
            PRIVACY_DIR / "synthetic_entity_matches.csv", index=False
        )

        synthetic_privacy_frame = pd.DataFrame({"output_text": synthetic_privacy_texts})
        phrase_path = PRIVACY_DIR / "entity_context_spans.csv"
        search_and_compute_EPO(
            synth_file=synthetic_privacy_frame,
            reference_texts=real_privacy_texts,
            synth_phrase_file_path=str(phrase_path),
            entity_patterns=entity_candidates,
            max_window_len=PRIVACY_MAX_CONTEXT_WINDOW,
            text_field="output_text",
        )
        phrase_frame = pd.read_csv(phrase_path)
        if phrase_frame.empty:
            overlap_counts = pd.DataFrame(columns=[
                "Context Length", "candidate_phrase_count",
                "reference_overlap_count", "reference_overlap_rate"
            ])
        else:
            phrase_frame = phrase_frame.drop_duplicates(
                subset=["Entity", "Phrase", "Context Length"]
            ).copy()
            phrase_frame["reference_match"] = phrase_frame["Phrase"].map(
                lambda phrase: any(phrase in reference for reference in real_privacy_texts)
            )
            phrase_frame.to_csv(PRIVACY_DIR / "entity_context_spans_with_matches.csv", index=False)
            overlap_counts = phrase_frame.groupby("Context Length").agg(
                candidate_phrase_count=("Phrase", "size"),
                reference_overlap_count=("reference_match", "sum"),
            ).reset_index()
            overlap_counts["reference_overlap_rate"] = (
                overlap_counts["reference_overlap_count"]
                / overlap_counts["candidate_phrase_count"]
            )
        overlap_counts.to_csv(PRIVACY_DIR / "entity_phrase_overlap_counts.csv", index=False)
    else:
        average_entity_matches = 0.0
        leaked_entity_union = set()
        overlap_counts = pd.DataFrame(columns=[
            "Context Length", "candidate_phrase_count",
            "reference_overlap_count", "reference_overlap_rate"
        ])
        overlap_counts.to_csv(PRIVACY_DIR / "entity_phrase_overlap_counts.csv", index=False)

    privacy_summary = {
        "toolkit_functions": ["entity_leakage", "search_and_compute_EPO"],
        "synthetic_documents": len(synthetic_privacy_texts),
        "real_reference_documents": len(real_privacy_texts),
        "candidate_named_entities": len(entity_candidates),
        "candidate_entities_matched_at_least_once": len(leaked_entity_union),
        "average_candidate_entity_matches_per_synthetic_document": float(average_entity_matches),
        "candidate_entity_source": "spaCy NER over the full available real ENNI reference corpus",
        "verified_private_entity_list_available": False,
        "interpretation": (
            "Reference named-entity and context-span overlap diagnostic only; "
            "not a verified privacy-leakage estimate."
        ),
        "canary_analysis_run": False,
        "canary_reason": "No known canaries were planted during generator training.",
        "epo_case_sensitive_as_implemented_by_package": True,
    }
    (PRIVACY_DIR / "privacy_summary.json").write_text(
        json.dumps(privacy_summary, indent=2) + "\n"
    )
    print(privacy_summary)
    display(overlap_counts)
else:
    print("Privacy analysis skipped because RUN_PRIVACY_ANALYSIS is False.")


## Automated text-quality evaluation

These toolkit outputs compare balanced synthetic and real corpora. They are not clinical validity measures. Perplexity and Fréchet distance can run on CPU; MAUVE should be enabled only in a GPU runtime.

In [ ]:
import dataclasses
import inspect
from synthtexteval.eval.text_quality import metrics as text_quality_metrics
from synthtexteval.eval.text_quality.arguments import (
    Arguments as TextQualityArguments,
    FrechetArgs,
    LMArgs,
    MauveArgs,
)

# Resolve the evaluator class against the actual pinned source loaded by this runtime.
QualityEvalClass = getattr(text_quality_metrics, "QualEval", None)
if QualityEvalClass is None:
    QualityEvalClass = getattr(text_quality_metrics, "TextQualityEval", None)
if QualityEvalClass is None:
    raise ImportError(
        "Neither QualEval nor TextQualityEval exists in "
        f"{text_quality_metrics.__file__}. Inspect that module before continuing."
    )

quality_args = TextQualityArguments(
    frechet=FrechetArgs(source_text_column="source", ref_text_column="reference"),
    mauve=MauveArgs(source_text_column="source", ref_text_column="reference"),
    perplexity=LMArgs(source_text_column="source", ref_text_column="reference"),
)
quality_evaluator = QualityEvalClass(quality_args)

# Save the exact evaluator class, argument values and MAUVE wrapper source so the result can
# be reconstructed even if the upstream package changes later.
try:
    quality_args_serialized = dataclasses.asdict(quality_args)
except TypeError:
    quality_args_serialized = {
        key: vars(value) if hasattr(value, "__dict__") else str(value)
        for key, value in vars(quality_args).items()
    }
quality_method_manifest = {
    "evaluator_class": QualityEvalClass.__name__,
    "metrics_module": text_quality_metrics.__file__,
    "arguments": quality_args_serialized,
}
(RESULTS_DIR / "text_quality_configuration.json").write_text(
    json.dumps(quality_method_manifest, indent=2, default=str) + "\n"
)
try:
    (RESULTS_DIR / "synthtexteval_calculate_mauve_score_source.txt").write_text(
        inspect.getsource(QualityEvalClass.calculate_mauve_score), encoding="utf-8"
    )
except (OSError, TypeError):
    pass

quality_evaluator.calculate_perplexity(comparison_df)
quality_evaluator.calculate_fid_score(comparison_df)
if RUN_MAUVE:
    quality_evaluator.calculate_mauve_score(comparison_df)
quality_results = quality_evaluator.return_results()
quality_evaluator.print_metrics(quality_results)
(RESULTS_DIR / "text_quality_results.json").write_text(
    json.dumps(quality_results, indent=2, default=str) + "\n"
)


### MAUVE stability diagnostics

The supplementary MAUVE diagnostics now **inherit the exact featurizer configuration used by the main SynthTextEval MAUVE call** rather than silently using MAUVE's own defaults. In this pinned SynthTextEval revision the main configuration is normally `gpt2`, `max_text_length=1024`, and the configured device ID. The first full-corpus repeated run uses MAUVE's default seed (`25`), so it should closely reproduce the main SynthTextEval score.

To avoid repeatedly running the language-model featurizer, real and synthetic feature matrices are computed once, cached, and then reused for:

1. **Full-corpus seed stability:** same feature matrices, different clustering seeds.
2. **80% subsampling sensitivity:** subsets of the cached feature matrices, different samples and seeds.
3. **Class-conditional MAUVE:** SLI and TD are evaluated separately using the same cached features. This is diagnostic: if synthetic TD has substantially poorer distributional fidelity than synthetic SLI, that may help explain the synthetic-only classifier's asymmetric class performance.

These distributions are stability/sensitivity analyses, not formal confidence intervals. The notebook writes a parity check comparing the first direct full-corpus score to the main SynthTextEval MAUVE result.


In [ ]:
import hashlib
import re
import numpy as np
import pandas as pd
import mauve
from mauve.compute_mauve import get_features_from_input


def _corpus_digest(texts):
    digest = hashlib.sha256()
    for text in texts:
        digest.update(str(text).encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def _safe_model_name(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name))


def _get_or_cache_mauve_features(
    texts,
    label,
    featurize_model_name,
    max_text_length,
    device_id,
    verbose,
):
    cache_dir = RESULTS_DIR / "mauve_feature_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)

    digest = _corpus_digest(texts)
    cache_path = cache_dir / (
        f"{label}_{_safe_model_name(featurize_model_name)}_"
        f"len{max_text_length}_{digest}.npy"
    )

    if cache_path.exists():
        print("Loading cached MAUVE features:", cache_path)
        return np.load(cache_path), cache_path

    print(f"Featurizing {label} once with {featurize_model_name} ...")
    features = get_features_from_input(
        features=None,
        tokenized_texts=None,
        texts=texts,
        featurize_model_name=featurize_model_name,
        max_len=max_text_length,
        device_id=device_id,
        name=label,
        batch_size=1,
        verbose=verbose,
        use_float64=False,
    )
    np.save(cache_path, features)
    return features, cache_path


def repeated_mauve_from_features(
    real_features,
    synth_features,
    n_runs,
    sample_size,
    sampling_seed,
    label,
):
    n = min(len(real_features), len(synth_features))
    if sample_size is None:
        sample_size = n
    if not 0 < sample_size <= n:
        raise ValueError(f"sample_size must be between 1 and {n}, got {sample_size}.")
    if n_runs < 1:
        raise ValueError(f"n_runs must be positive, got {n_runs}.")

    rng = np.random.default_rng(sampling_seed)
    rows = []

    for run_index in range(n_runs):
        if sample_size == n:
            real_idx = np.arange(len(real_features))
            synth_idx = np.arange(len(synth_features))
        else:
            real_idx = rng.choice(len(real_features), size=sample_size, replace=False)
            synth_idx = rng.choice(len(synth_features), size=sample_size, replace=False)

        mauve_seed = MAUVE_STABILITY_BASE_SEED + run_index
        result = mauve.compute_mauve(
            p_features=real_features[real_idx],
            q_features=synth_features[synth_idx],
            seed=mauve_seed,
            verbose=False,
        )

        rows.append({
            "analysis": label,
            "run": run_index,
            "sampling_seed": sampling_seed,
            "mauve_seed": mauve_seed,
            "sample_size_per_distribution": sample_size,
            "mauve": float(result.mauve),
            "num_buckets": int(result.num_buckets),
        })

    frame = pd.DataFrame(rows)
    summary = {
        "analysis": label,
        "n_runs": int(len(frame)),
        "sample_size_per_distribution": int(sample_size),
        "full_balanced_corpus_size_per_distribution": int(n),
        "mean": float(frame["mauve"].mean()),
        "sd_sample": float(frame["mauve"].std(ddof=1)) if len(frame) > 1 else 0.0,
        "min": float(frame["mauve"].min()),
        "max": float(frame["mauve"].max()),
        "sampling_seed": int(sampling_seed),
        "mauve_seed_start": int(MAUVE_STABILITY_BASE_SEED),
        "interpretation": "stability/sensitivity diagnostic; not a formal confidence interval",
    }
    return frame, summary


if RUN_REPEATED_MAUVE:
    # Inherit the main SynthTextEval MAUVE configuration directly.
    main_mauve_args = quality_args.mauve
    mauve_feature_model = getattr(
        main_mauve_args, "model_name_featurizer", "gpt2-large"
    )
    mauve_max_length = int(
        getattr(main_mauve_args, "max_text_length", MAUVE_MAX_TEXT_LENGTH)
    )
    mauve_device_id = int(
        getattr(main_mauve_args, "device_id", MAUVE_DEVICE_ID)
    )
    mauve_verbose = bool(getattr(main_mauve_args, "verbose", False))

    exact_mauve_config = {
        "featurize_model_name": mauve_feature_model,
        "max_text_length": mauve_max_length,
        "device_id": mauve_device_id,
        "verbose": mauve_verbose,
        "first_seed": int(MAUVE_STABILITY_BASE_SEED),
        "library_defaults_for_clustering_and_divergence": True,
        "note": (
            "Feature-model settings are inherited from quality_args.mauve. "
            "When p_features/q_features are supplied, featurization is not repeated."
        ),
    }
    (RESULTS_DIR / "repeated_mauve_exact_configuration.json").write_text(
        json.dumps(exact_mauve_config, indent=2) + "\n"
    )
    print("Repeated MAUVE configuration:", exact_mauve_config)

    real_list = comparison_df["reference"].astype(str).tolist()
    synth_list = comparison_df["source"].astype(str).tolist()
    n_balanced = min(len(real_list), len(synth_list))

    real_features, real_feature_cache = _get_or_cache_mauve_features(
        real_list,
        "real",
        mauve_feature_model,
        mauve_max_length,
        mauve_device_id,
        mauve_verbose,
    )
    synth_features, synth_feature_cache = _get_or_cache_mauve_features(
        synth_list,
        "synthetic",
        mauve_feature_model,
        mauve_max_length,
        mauve_device_id,
        mauve_verbose,
    )

    if len(real_features) != len(real_list) or len(synth_features) != len(synth_list):
        raise RuntimeError("Cached MAUVE feature row counts do not match the current corpora.")

    stability_frames = []
    stability_summaries = []

    full_frame, full_summary = repeated_mauve_from_features(
        real_features=real_features,
        synth_features=synth_features,
        n_runs=REPEATED_MAUVE_N_RUNS,
        sample_size=n_balanced,
        sampling_seed=MAUVE_STABILITY_BASE_SEED,
        label="full_corpus_seed_stability",
    )
    stability_frames.append(full_frame)
    stability_summaries.append(full_summary)

    if MAUVE_SUBSAMPLE_FRACTION is not None:
        if not 0 < MAUVE_SUBSAMPLE_FRACTION < 1:
            raise ValueError("MAUVE_SUBSAMPLE_FRACTION must be between 0 and 1, or None.")
        subsample_size = max(2, int(np.floor(n_balanced * MAUVE_SUBSAMPLE_FRACTION)))
        subsample_frame, subsample_summary = repeated_mauve_from_features(
            real_features=real_features,
            synth_features=synth_features,
            n_runs=REPEATED_MAUVE_N_RUNS,
            sample_size=subsample_size,
            sampling_seed=MAUVE_STABILITY_BASE_SEED,
            label="subsample_sensitivity",
        )
        subsample_summary["subsample_fraction"] = float(MAUVE_SUBSAMPLE_FRACTION)
        stability_frames.append(subsample_frame)
        stability_summaries.append(subsample_summary)

    repeated_mauve_frame = pd.concat(stability_frames, ignore_index=True)
    repeated_mauve_frame.to_csv(RESULTS_DIR / "repeated_mauve_scores.csv", index=False)

    # The first full-corpus run uses seed=25, MAUVE's default seed used by the wrapper
    # unless the wrapper overrides it. With matching feature configuration it should
    # closely reproduce the main SynthTextEval result.
    main_score = None
    if isinstance(quality_results, dict) and quality_results.get("mauve"):
        main_score = float(quality_results["mauve"][-1])

    first_direct_score = float(
        full_frame.sort_values("run").iloc[0]["mauve"]
    )
    parity = {
        "main_synthtexteval_mauve": main_score,
        "first_direct_full_corpus_mauve": first_direct_score,
        "absolute_difference": (
            abs(main_score - first_direct_score)
            if main_score is not None else None
        ),
        "same_featurizer_model": mauve_feature_model,
        "same_max_text_length": mauve_max_length,
        "first_direct_seed": int(MAUVE_STABILITY_BASE_SEED),
    }
    (RESULTS_DIR / "mauve_wrapper_parity_check.json").write_text(
        json.dumps(parity, indent=2) + "\n"
    )
    print("MAUVE wrapper parity check:", parity)

    # Class-conditional MAUVE uses the already-computed features, so it adds little
    # runtime and can help diagnose asymmetric SLI/TD synthetic fidelity.
    class_rows = []
    real_groups = reference_texts[GROUP_COLUMN].astype(str).reset_index(drop=True)
    synth_groups = synthetic_texts[GROUP_COLUMN].astype(str).reset_index(drop=True)

    for group_name in COMPARISON_GROUPS:
        real_idx = np.flatnonzero(real_groups.eq(group_name).to_numpy())
        synth_idx = np.flatnonzero(synth_groups.eq(group_name).to_numpy())
        n_group = min(len(real_idx), len(synth_idx))
        if n_group < 2:
            continue

        result = mauve.compute_mauve(
            p_features=real_features[real_idx[:n_group]],
            q_features=synth_features[synth_idx[:n_group]],
            seed=MAUVE_STABILITY_BASE_SEED,
            verbose=False,
        )
        class_rows.append({
            "group": group_name,
            "n_real": int(len(real_idx)),
            "n_synthetic": int(len(synth_idx)),
            "n_used_per_distribution": int(n_group),
            "mauve": float(result.mauve),
            "num_buckets": int(result.num_buckets),
            "seed": int(MAUVE_STABILITY_BASE_SEED),
        })

    class_conditional_mauve = pd.DataFrame(class_rows)
    class_conditional_mauve.to_csv(
        RESULTS_DIR / "class_conditional_mauve.csv", index=False
    )

    for summary in stability_summaries:
        summary.update({
            "featurize_model_name": mauve_feature_model,
            "max_text_length": mauve_max_length,
            "device_id": mauve_device_id,
            "real_feature_cache": str(real_feature_cache),
            "synthetic_feature_cache": str(synth_feature_cache),
        })
    (RESULTS_DIR / "repeated_mauve_summary.json").write_text(
        json.dumps(stability_summaries, indent=2) + "\n"
    )

    display(repeated_mauve_frame)
    display(
        repeated_mauve_frame.groupby("analysis")["mauve"]
        .agg(["count", "mean", "std", "min", "max"]).reset_index()
    )
    print("Class-conditional MAUVE")
    display(class_conditional_mauve)
else:
    print("Repeated MAUVE skipped because RUN_REPEATED_MAUVE is False.")


## Save results to Drive, create a verified ZIP, and download locally

The export is **compact by default**: trained BERT `model/` and trainer/tester directories are omitted because they can make the archive several gigabytes. Set `EXPORT_INCLUDE_MODEL_CHECKPOINTS = True` in the configuration cell if checkpoint weights must be archived too.

The first cell below writes a run/environment manifest, copies the result bundle to Google Drive, creates a ZIP, checks the ZIP integrity and SHA-256 checksum, copies the ZIP to Drive, and starts the browser download. The browser's local Downloads folder is not visible to the Colab Python runtime, so local completion cannot be programmatically verified from the server.

**Only after the ZIP is visibly present on your computer should you run the final cell.** It asks for an explicit confirmation string before calling `runtime.unassign()`.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import subprocess
import zipfile

from google.colab import drive, files

drive.mount("/content/drive", force_remount=False)
DRIVE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

export_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
export_name = f"{EXPORT_ARCHIVE_PREFIX}_{export_timestamp}"
DRIVE_RUN_DIR = DRIVE_EXPORT_ROOT / export_name
LOCAL_EXPORT_DIR = Path("/content") / export_name
LOCAL_ZIP_PATH = Path("/content") / f"{export_name}.zip"
DRIVE_ZIP_PATH = DRIVE_EXPORT_ROOT / LOCAL_ZIP_PATH.name

def git_head(repo_dir):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception:
        return None

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

def sha256_path(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

run_manifest = {
    "export_timestamp_utc": export_timestamp,
    "evaluation_run_id": RESULTS_DIR.name,
    "project_root": str(PROJECT_ROOT),
    "results_dir": str(RESULTS_DIR),
    "icl_pilot_git_head": git_head(PROJECT_ROOT),
    "synthtexteval_git_head": git_head(SYNTHTEXTEVAL_DIR),
    "synthtexteval_requested_revision": SYNTHTEXTEVAL_REVISION,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in [
            "SynthTextEval", "transformers", "torch", "datasets", "mauve-text",
            "spacy", "scikit-learn", "pandas", "numpy"
        ]
    },
    "configuration": {
        "comparison_cohorts": list(COMPARISON_COHORTS),
        "comparison_groups": list(COMPARISON_GROUPS),
        "sample_seed": SAMPLE_SEED,
        "max_samples_per_group": MAX_SAMPLES_PER_GROUP,
        "classification_rounds": list(CLASSIFICATION_ROUNDS),
        "classifier_model": CLASSIFIER_MODEL,
        "label_mapping": LABEL_MAPPING,
        "run_classification": RUN_CLASSIFICATION,
        "run_gillam_external_test": RUN_GILLAM_EXTERNAL_TEST,
        "run_child_level_classification": RUN_CHILD_LEVEL_CLASSIFICATION,
        "run_privacy_analysis": RUN_PRIVACY_ANALYSIS,
        "run_mauve": RUN_MAUVE,
        "run_repeated_mauve": RUN_REPEATED_MAUVE,
        "repeated_mauve_n_runs": REPEATED_MAUVE_N_RUNS,
        "mauve_subsample_fraction": MAUVE_SUBSAMPLE_FRACTION,
        "mauve_stability_base_seed": MAUVE_STABILITY_BASE_SEED,
        "export_include_model_checkpoints": EXPORT_INCLUDE_MODEL_CHECKPOINTS,
    },
    "input_files": {
        str(TEXT_RECORDS_CSV): {
            "sha256": sha256_path(TEXT_RECORDS_CSV),
            "size_bytes": TEXT_RECORDS_CSV.stat().st_size,
        },
        str(GILLAM_EXTERNAL_TEXT_CSV): {
            "sha256": sha256_path(GILLAM_EXTERNAL_TEXT_CSV),
            "size_bytes": GILLAM_EXTERNAL_TEXT_CSV.stat().st_size,
        },
    },
}
(RESULTS_DIR / "run_environment_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, default=str) + "\n"
)

# Build a clean local export tree. The original RESULTS_DIR remains untouched.
if LOCAL_EXPORT_DIR.exists():
    shutil.rmtree(LOCAL_EXPORT_DIR)

excluded_dir_names = {"__pycache__"}
if not EXPORT_INCLUDE_MODEL_CHECKPOINTS:
    excluded_dir_names.update({"model", "trainer", "tester", "external_gillam_tester"})

def export_ignore(directory, names):
    ignored = []
    for name in names:
        path = Path(directory) / name
        if path.is_dir() and name in excluded_dir_names:
            ignored.append(name)
    return ignored

shutil.copytree(RESULTS_DIR, LOCAL_EXPORT_DIR / "results", ignore=export_ignore)

# Add a checksum manifest for every portable result file.
exported_files = []
for path in sorted((LOCAL_EXPORT_DIR / "results").rglob("*")):
    if path.is_file():
        exported_files.append({
            "path": str(path.relative_to(LOCAL_EXPORT_DIR)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_path(path),
        })
(LOCAL_EXPORT_DIR / "portable_export_file_manifest.json").write_text(
    json.dumps(exported_files, indent=2) + "\n"
)

# Copy the portable directory to Drive and verify it exists before zipping.
if DRIVE_RUN_DIR.exists():
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_RUN_DIR)
assert (DRIVE_RUN_DIR / "results" / "run_environment_manifest.json").exists()

# Create the local ZIP from the portable export directory.
if LOCAL_ZIP_PATH.exists():
    LOCAL_ZIP_PATH.unlink()
shutil.make_archive(
    str(LOCAL_ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=LOCAL_EXPORT_DIR.parent,
    base_dir=LOCAL_EXPORT_DIR.name,
)
assert LOCAL_ZIP_PATH.exists() and LOCAL_ZIP_PATH.stat().st_size > 0

with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as archive:
    bad_member = archive.testzip()
    assert bad_member is None, f"ZIP integrity check failed at: {bad_member}"
    zip_member_count = len(archive.infolist())

local_zip_sha256 = sha256_path(LOCAL_ZIP_PATH)
shutil.copy2(LOCAL_ZIP_PATH, DRIVE_ZIP_PATH)
assert DRIVE_ZIP_PATH.exists()
drive_zip_sha256 = sha256_path(DRIVE_ZIP_PATH)
assert local_zip_sha256 == drive_zip_sha256, "Drive ZIP checksum differs from local ZIP."

download_state = {
    "drive_result_directory": str(DRIVE_RUN_DIR),
    "drive_zip": str(DRIVE_ZIP_PATH),
    "local_zip_for_browser_download": str(LOCAL_ZIP_PATH),
    "zip_size_bytes": int(LOCAL_ZIP_PATH.stat().st_size),
    "zip_members": int(zip_member_count),
    "zip_sha256": local_zip_sha256,
    "server_side_checks_passed": True,
    "browser_download_confirmed": False,
}
(DRIVE_RUN_DIR / "download_state_before_local_confirmation.json").write_text(
    json.dumps(download_state, indent=2) + "\n"
)

print(json.dumps(download_state, indent=2))
print("\nServer-side Drive copy, ZIP integrity, and checksum checks passed.")
print("Starting browser download now. Do NOT run the final cell until the ZIP is visibly present on your computer.")
files.download(str(LOCAL_ZIP_PATH))

### Final step: confirm the local download, then release the Colab runtime

Run this cell **manually after checking your computer's Downloads folder**. Typing anything other than the exact confirmation string leaves the runtime assigned.

In [ ]:
# This MUST remain the final executable cell in the notebook.
confirmation = input(
    f"Confirm that {LOCAL_ZIP_PATH.name} is fully downloaded on your computer. "
    "Type exactly DOWNLOADED to unassign this Colab runtime: "
).strip()

if confirmation != "DOWNLOADED":
    raise RuntimeError(
        "Local download was not explicitly confirmed. Runtime remains assigned. "
        "Verify the ZIP in your Downloads folder, then rerun this final cell."
    )

# Recheck the durable Drive copy immediately before releasing compute.
assert DRIVE_ZIP_PATH.exists(), f"Drive ZIP is missing: {DRIVE_ZIP_PATH}"
assert sha256_path(DRIVE_ZIP_PATH) == local_zip_sha256, "Drive ZIP checksum changed."

confirmed_state = dict(download_state)
confirmed_state["browser_download_confirmed"] = True
confirmed_state["confirmation_method"] = "manual user verification in browser/local Downloads folder"
(DRIVE_RUN_DIR / "download_state_confirmed.json").write_text(
    json.dumps(confirmed_state, indent=2) + "\n"
)
print("Local download confirmed and Drive archive rechecked. Unassigning Colab runtime.")

from google.colab import runtime
runtime.unassign()